In [99]:
import os
import re
import pandas as pd

In [100]:
# =========================
# 1. НАСТРОЙКИ И ПУТИ
# =========================

DATA_DIR = "data"

QUESTIONS_FILE = os.path.join(DATA_DIR, "user_questions.xlsx")
ABUSIVE_WORDS_FILE = os.path.join(DATA_DIR, "ru_abusive_words.txt")
CURSE_WORDS_FILE = os.path.join(DATA_DIR, "ru_curse_words.txt")

In [101]:
# =========================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================

RUSSIAN_LIGHT_SUFFIXES = [
    "иями", "ями", "ами", "иях", "ях", "ого", "ему", "ому", "ими", "ыми",
    "ее", "ие", "ые", "ое", "ей", "ий", "ый", "ой", "ем", "им", "ым", "ом",
    "его", "их", "ых", "ую", "юю", "ая", "яя", "ою", "ею", "ах", "ях", "ам",
    "ям", "ов", "ев", "ия", "ья", "ию", "ью", "ие", "ье", "ии", "ьи",
    "а", "я", "ы", "и", "е", "у", "ю", "о"
]

def normalize_text(text: str) -> str:
    """
    Базовая нормализация текста:
    - lower
    - ё -> е
    - убираем лишние пробелы
    """
    if not isinstance(text, str):
        return ""

    text = text.lower().strip()
    text = text.replace("ё", "е")
    text = re.sub(r"\s+", " ", text)
    return text


def normalize_match_text(text: str) -> str:
    """
    Нормализация для сопоставления:
    - lower
    - ё -> е
    - кавычки / пунктуацию -> пробел
    - дефисы и слэши -> пробел
    """
    text = normalize_text(str(text))
    text = re.sub(r"[«»\"“”]", " ", text)
    text = re.sub(r"[-–—/]", " ", text)
    text = re.sub(r"[^а-яa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def light_stem(word: str) -> str:
    """
    Очень лёгкий стемминг без внешних библиотек.
    Нужен не для идеальной лемматизации, а чтобы ловить:
    - внешнеэкономическая / внешнеэкономической
    - бюджет / бюджетных
    - программа / программе
    """
    word = normalize_match_text(word)
    if len(word) <= 3:
        return word

    for suffix in sorted(RUSSIAN_LIGHT_SUFFIXES, key=len, reverse=True):
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)]
    return word


def simple_tokenize(text: str) -> list:
    """
    Простая токенизация:
    оставляем буквы/цифры и разбиваем по словам.
    """
    text = normalize_match_text(text)
    tokens = re.findall(r"[а-яa-z0-9]+", text)
    return tokens


def stem_tokenize(text: str) -> list:
    return [light_stem(tok) for tok in simple_tokenize(text) if tok]


def stem_string(text: str) -> str:
    return " ".join(stem_tokenize(text))


def load_words_from_txt(file_path: str) -> set:
    """
    Загружаем слова из txt в set.
    Пустые строки игнорируем.
    """
    words = set()

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            word = normalize_text(line.strip())
            if word:
                words.add(word)

    return words

In [102]:
# =========================
# 3. БЛОК ВХОДА
# =========================

def load_questions(file_path: str) -> pd.DataFrame:
    """
    Загружает вопросы пользователей из Excel.
    Ожидается колонка 'question'.
    """
    df = pd.read_excel(file_path)

    if "question" not in df.columns:
        raise ValueError("В файле с вопросами должна быть колонка 'question'")

    df["question"] = df["question"].fillna("").astype(str)
    return df

In [103]:
# =========================
# 4. БЛОК МОДЕРАЦИИ
# =========================

TOXIC_STEM_HINTS = {
    "бляд", "бля", "пизд", "ху", "еб", "сук", "жоп", "хрен", "нахрен",
    "долбан", "черт", "чертвозьм", "мудак", "гандон"
}

TOXIC_REGEX_PATTERNS = [
    r"\bнахрен\b",
    r"\bкакого\s+хрена\b",
    r"\bчерт\s+возьми\b",
    r"\bоху[еияй]\w*",
    r"\bдолбан\w*",
]

def check_empty_or_too_short(text: str) -> bool:
    text = normalize_text(text)
    return len(text) == 0


def check_too_long(text: str, max_len: int = 500) -> bool:
    return len(text) > max_len


def contains_bad_words(text: str, bad_words: set) -> tuple:
    """
    Проверяем токсичность мягче, чем просто по точному словарю:
    1) точное совпадение по токенам
    2) подстроки
    3) стемы
    4) несколько дополнительных regex-паттернов
    """
    normalized = normalize_text(text)
    normalized_match = normalize_match_text(text)
    tokens = simple_tokenize(normalized_match)
    stem_tokens_q = stem_tokenize(normalized_match)

    found_words = set()

    # 1) точное совпадение по токенам
    for token in tokens:
        if token in bad_words:
            found_words.add(token)

    # 2) проверка по подстроке
    for bad_word in bad_words:
        bad_word = normalize_text(bad_word)
        if len(bad_word) >= 4 and bad_word in normalized:
            found_words.add(bad_word)

    # 3) проверка по стемам
    bad_stems = {light_stem(w) for w in bad_words if light_stem(w)}
    for tok in stem_tokens_q:
        if tok in bad_stems or tok in TOXIC_STEM_HINTS:
            found_words.add(tok)

    # 4) дополнительные паттерны
    for pattern in TOXIC_REGEX_PATTERNS:
        if re.search(pattern, normalized_match):
            found_words.add(pattern)

    found_words = sorted(found_words)
    return len(found_words) > 0, found_words


def moderate_question(question: str, bad_words: set) -> dict:
    if check_empty_or_too_short(question):
        return {
            "status": "empty",
            "allow_to_continue": False,
            "comment": "Пустой вопрос",
            "found_bad_words": []
        }

    if check_too_long(question):
        return {
            "status": "too_long",
            "allow_to_continue": False,
            "comment": "Слишком длинный вопрос",
            "found_bad_words": []
        }

    is_toxic, found_words = contains_bad_words(question, bad_words)

    if is_toxic:
        return {
            "status": "toxic",
            "allow_to_continue": False,
            "comment": "Обнаружены нежелательные слова",
            "found_bad_words": found_words
        }

    return {
        "status": "ok",
        "allow_to_continue": True,
        "comment": "Вопрос прошёл модерацию",
        "found_bad_words": []
    }

In [104]:
# =========================
# 5. ЗАПУСК
# =========================

def main():
    abusive_words = load_words_from_txt(ABUSIVE_WORDS_FILE)
    curse_words = load_words_from_txt(CURSE_WORDS_FILE)
    all_bad_words = abusive_words.union(curse_words)

    print(f"Загружено нежелательных слов: {len(all_bad_words)}")

    questions_df = load_questions(QUESTIONS_FILE)

    results = []

    for _, row in questions_df.iterrows():
        question = row["question"]
        moderation_result = moderate_question(question, all_bad_words)

        results.append({
            "question": question,
            "status": moderation_result["status"],
            "allow_to_continue": moderation_result["allow_to_continue"],
            "comment": moderation_result["comment"],
            "found_bad_words": ", ".join(moderation_result["found_bad_words"])
        })

    result_df = pd.DataFrame(results)

    print("\nРезультаты модерации:")
    print(result_df.head(10))

    return result_df

In [105]:
moderated_df = main()

Загружено нежелательных слов: 2274

Результаты модерации:
                                            question status  \
0  Какие ЕГЭ нужны для поступления на программу «...     ok   
1  А на платное на «Анализ данных и искусственный...     ok   
2      Сколько стоит обучение на бизнес-информатике?     ok   
3  Какой проходной балл был в 2024 году на бизнес...     ok   
4  Сколько бюджетных мест в 2025 году на программ...     ok   
5            Сколько платных мест на веб-разработке?     ok   
6      Есть ли бюджет на программе «Веб-разработка»?     ok   
7  Сколько лет учиться на программе «Внешнеэконом...     ok   
8  К какому мегакластеру относится программа «Мар...     ok   
9  Какие требования по ЕГЭ на платное у программы...     ok   

   allow_to_continue                  comment found_bad_words  
0               True  Вопрос прошёл модерацию                  
1               True  Вопрос прошёл модерацию                  
2               True  Вопрос прошёл модерацию           

# Определение токсичности

In [106]:
from transformers import pipeline

In [107]:
toxicity_classifier = pipeline(
    task="text-classification",
    model="fasherr/toxicity_rubert",
    tokenizer="fasherr/toxicity_rubert",
    truncation=True,
    max_length=512
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7648.98it/s]


In [108]:
def check_toxicity(question: str, toxicity_threshold: float = 0.75) -> dict:
    """
    Проверка токсичности через отдельную модель.

    Возвращает:
    - toxicity_label: toxic / non_toxic
    - toxicity_score: уверенность модели
    - allow_after_toxicity: можно ли пускать дальше
    """
    pred = toxicity_classifier(question)[0]

    raw_label = str(pred["label"]).lower()
    raw_score = float(pred["score"])

    # На случай разных схем названий меток
    toxic_markers = ["toxic", "1", "label_1"]

    is_toxic = any(marker in raw_label for marker in toxic_markers)

    if is_toxic and raw_score >= toxicity_threshold:
        return {
            "toxicity_label": "toxic",
            "toxicity_score": round(raw_score, 4),
            "allow_after_toxicity": False
        }

    return {
        "toxicity_label": "non_toxic",
        "toxicity_score": round(raw_score, 4),
        "allow_after_toxicity": True
    }

In [109]:
def run_toxicity_filter(df: pd.DataFrame, toxicity_threshold: float = 0.75) -> pd.DataFrame:
    df = df.copy()

    toxicity_labels = []
    toxicity_scores = []
    allow_after_toxicity_list = []

    for _, row in df.iterrows():
        question = row["question"]
        allow_to_continue = row["allow_to_continue"]

        if not allow_to_continue:
            toxicity_labels.append("blocked_by_moderation")
            toxicity_scores.append(None)
            allow_after_toxicity_list.append(False)
            continue

        result = check_toxicity(question, toxicity_threshold=toxicity_threshold)

        toxicity_labels.append(result["toxicity_label"])
        toxicity_scores.append(result["toxicity_score"])
        allow_after_toxicity_list.append(result["allow_after_toxicity"])

    df["toxicity_label"] = toxicity_labels
    df["toxicity_score"] = toxicity_scores
    df["allow_after_toxicity"] = allow_after_toxicity_list

    return df

In [110]:
toxicity_df = run_toxicity_filter(moderated_df, toxicity_threshold=0.75)

# Определение типа вопроса

Он должен понять: что именно спрашивает пользователь.

Например, вопрос может быть:

FAQ — типовой вопрос с готовым ответом;  
табличный запрос — стоимость, баллы, ЕГЭ, бюджетные места;  
общий текстовый вопрос — например, про мегакластеры, особенности направления, партнёров;     
непонятный вопрос, где не хватает информации.

In [116]:
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

In [117]:
FAQ_FILE = os.path.join(DATA_DIR, "Database.xlsx")
GENERAL_FILE = os.path.join(DATA_DIR, "Database-2.xlsx")
PROGRAM_FILE = os.path.join(DATA_DIR, "all_program.xlsx")

In [118]:
INTENT_MODEL_NAME = "cointegrated/rubert-tiny2"
intent_model = SentenceTransformer(INTENT_MODEL_NAME)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 4539.83it/s]
BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [119]:
def load_faq_examples(faq_file: str) -> list:
    faq_df = pd.read_excel(faq_file)

    faq_questions = (
        faq_df["Question"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    faq_questions = [q for q in faq_questions if q]
    return faq_questions

In [120]:
def load_general_examples(general_file: str) -> list:
    general_df = pd.read_excel(general_file)

    headers = (
        general_df["header"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    examples = []

    for h in headers:
        if not h:
            continue

        examples.append(h)

        # если заголовок сам не вопрос, добавим несколько шаблонов
        examples.append(f"Расскажи про {h}")
        examples.append(f"Что значит {h}?")

    # убираем дубли
    examples = list(dict.fromkeys(examples))
    return examples

In [121]:
def load_table_examples(program_file: str, max_programs: int = 30) -> list:
    program_df = pd.read_excel(program_file)

    program_names = (
        program_df["program"]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    program_names = list(dict.fromkeys(program_names))[:max_programs]

    examples = []

    for program in program_names:
        examples.extend([
            f"Сколько стоит обучение на программе {program}?",
            f"Какие ЕГЭ нужны для поступления на программу {program}?",
            f"Сколько бюджетных мест на программе {program}?",
            f"Сколько платных мест на программе {program}?",
            f"Какой проходной балл на программе {program}?",
            f"Какая форма обучения у программы {program}?",
            f"Сколько лет длится обучение на программе {program}?",
            f"Какие треки есть на программе {program}?",
            f"Какой институт отвечает за программу {program}?"
        ])

    return examples

In [122]:
faq_examples = load_faq_examples(FAQ_FILE)
general_examples = load_general_examples(GENERAL_FILE)
table_examples = load_table_examples(PROGRAM_FILE, max_programs=30)

INTENT_EXAMPLES = {
    "faq": faq_examples,
    "general": general_examples,
    "table": table_examples
}

In [123]:
intent_label_texts = []
intent_label_names = []

for label, examples in INTENT_EXAMPLES.items():
    for example in examples:
        intent_label_texts.append(example)
        intent_label_names.append(label)

In [124]:
intent_example_embeddings = intent_model.encode(
    intent_label_texts,
    convert_to_tensor=True,
    normalize_embeddings=True
)

In [125]:
from difflib import SequenceMatcher

In [126]:
# Более строгая доменная логика:
# 1) расширяем доменные примеры реальными FAQ / general / table вопросами
# 2) добавляем явные сильные маркеры
# 3) отдельно описываем поддерживаемые и неподдерживаемые атрибуты программы

DOMAIN_BASE_EXAMPLES = [
    "Какие ЕГЭ нужны для поступления на программу?",
    "Сколько стоит обучение на программе?",
    "Какой проходной балл на программу?",
    "Сколько бюджетных мест на программе?",
    "Сколько платных мест на программе?",
    "Что такое мегакластер?",
    "Чем отличаются мегакластеры?",
    "Какие есть образовательные программы?",
    "В каком институте реализуется программа?",
    "Какая длительность обучения на программе?",
    "Какие треки есть на программе?",
    "Чему учат на программе?",
    "Какие навыки получает выпускник?",
    "Какие секции есть в Академии?",
    "Есть ли у студентов лаборатории?",
    "Где находится кампус Академии?",
    "Что такое ЯДРО в образовательных программах?",
    "Есть ли военный учебный центр?",
    "Сколько лет действует олимпиада для БВИ?",
    "Академия и филиалы это один вуз?",
    "Можно ли изучать второй иностранный язык?",
]

DOMAIN_EXAMPLES = list(dict.fromkeys(
    DOMAIN_BASE_EXAMPLES
    + faq_examples[:250]
    + general_examples[:250]
    + table_examples[:180]
))

HARD_NEGATIVE_EXAMPLES = [
    "Какая сегодня погода?",
    "Напиши код на Python",
    "Что посмотреть вечером?",
    "Как приготовить пасту?",
    "Что такое институт брака?",
    "Какая программа лучше для монтажа видео?",
    "Сколько стоит обучение в автошколе?",
    "Какие экзамены нужны для поступления в колледж?",
    "Как подготовиться к ЕГЭ по математике?",
    "Куда поступать после 11 класса?",
    "Как выбрать профессию после школы?",
    "Какой университет лучше в Москве?",
    "Сколько стоит онлайн курс по аналитике?",
    "Какая программа тренировок лучше?",
    "Что нужно знать перед ЕГЭ?",
    "Как купить билет на поезд?",
    "Сколько стоит айфон?",
    "Кто выиграл Лигу чемпионов?",
]

DOMAIN_STRONG_KEYWORDS = [
    "поступ", "абитури", "егэ", "экзамен", "проходн", "балл", "бюдж", "платн",
    "контракт", "мест", "стоимост", "мегакластер", "направлен", "прием",
    "приемн", "вступит", "олимпиад", "бви", "филиал", "фотограф",
    "кампус", "корпус", "лаборатор", "секци", "военн", "ядро", "общежити",
    "стипенд", "иностранн", "академ"
]

DOMAIN_WEAK_KEYWORDS = [
    "обучен", "учеб", "программ", "институт", "трек", "навык",
    "выпускник", "специальност", "факультет", "партнер", "партнерств",
    "практик", "проект", "хакатон", "карьер", "проживан"
]

# Явные правила на поддерживаемые табличные поля
EXACT_FIELD_RULES = [
    {"field_name": "eges_budget", "patterns": ["какие егэ нужны на бюджет", "егэ на бюджет", "экзамены на бюджет"]},
    {"field_name": "eges_contract", "patterns": ["какие егэ нужны на платное", "егэ на платное", "егэ на контракт", "экзамены на платное"]},
    {"field_name": "budget_2025", "patterns": ["бюджетные места", "места на бюджете", "есть ли бюджет", "есть бюджет", "бюджет"]},
    {"field_name": "contract_2025", "patterns": ["платные места", "контрактные места", "места на платном обучении", "мест на платное"]},
    {"field_name": "cost", "patterns": ["сколько стоит", "стоимость обучения", "стоимость", "цена программы", "цена обучения"]},
    {"field_name": "pass_2024", "patterns": ["проходной балл", "какой проходной балл", "баллы для поступления", "какой балл"]},
    {"field_name": "edu_years", "patterns": ["сколько лет учиться", "длительность обучения", "срок обучения"]},
    {"field_name": "edu_form", "patterns": ["форма обучения", "очная", "заочная", "дистанционная"]},
    {"field_name": "megacluster", "patterns": ["какой мегакластер", "к какому мегакластеру относится", "мегакластер программы"]},
    {"field_name": "tracks", "patterns": ["какие есть треки", "профили программы", "специализации программы"]},
    {"field_name": "institute", "patterns": ["какой институт", "какая школа", "где реализуется программа"]},
    {"field_name": "major", "patterns": ["направление подготовки", "код направления"]},
    {"field_name": "desc", "patterns": ["о чем программа", "описание программы", "что изучают на программе"]},
    {"field_name": "skills", "patterns": ["чему научусь", "какие навыки получу", "компетенции выпускника"]},
]

UNSUPPORTED_PROGRAM_FIELD_RULES = {
    "salary": ["зарплат", "средняя зарплата"],
    "competition": ["конкурс на место", "конкурс"],
    "deadline": ["до какого числа", "срок подачи", "когда подать", "дедлайн"],
    "dorm": ["общежити", "проживан"],
    "avg_ege": ["средний балл егэ", "средний балл по русскому", "средний балл"],
    "employment": ["трудоустрой", "процент трудоустройства"],
    "stipend": ["стипенд"],
    "ranking": ["рейтинг"],
    "applications": ["подали заявления", "сколько заявлений", "сколько человек подали"],
}

In [127]:
# Каталог сущностей из all_program.xlsx.
# Эти сущности намного полезнее общих слов типа "программа" или "обучение".

program_meta_df = pd.read_excel(PROGRAM_FILE).fillna("")

def clean_entity_value(text: str) -> str:
    text = normalize_text(str(text))
    text = re.sub(r"[«»\"“”]", " ", text)
    text = re.sub(r"[-–—/]", " ", text)
    text = re.sub(r"[^а-яa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_multivalue_text(text: str) -> list:
    text = str(text)
    parts = re.split(r"[\n;,]+", text)
    parts = [clean_entity_value(p) for p in parts]
    parts = [p for p in parts if p and len(p) >= 3]
    return parts

def build_entity_catalog(df: pd.DataFrame) -> dict:
    catalog = {
        "program": set(),
        "megacluster": set(),
        "institute": set(),
        "major": set(),
        "track": set()
    }

    for value in df["program"].tolist():
        val = clean_entity_value(value)
        if len(val) >= 4:
            catalog["program"].add(val)

    for value in df["megacluster"].tolist():
        val = clean_entity_value(value)
        if len(val) >= 4:
            catalog["megacluster"].add(val)

    for value in df["institute"].tolist():
        val = clean_entity_value(value)
        if len(val) >= 3:
            catalog["institute"].add(val)

    for value in df["major"].tolist():
        val = clean_entity_value(value)
        if len(val) >= 5:
            catalog["major"].add(val)

    if "tracks" in df.columns:
        for value in df["tracks"].tolist():
            for part in split_multivalue_text(value):
                if len(part) >= 4:
                    catalog["track"].add(part)

    # более длинные сущности ищем первыми
    for key in catalog:
        catalog[key] = sorted(catalog[key], key=len, reverse=True)

    return catalog

ENTITY_CATALOG = build_entity_catalog(program_meta_df)



In [128]:
domain_example_embeddings = intent_model.encode(
    DOMAIN_EXAMPLES,
    convert_to_tensor=True,
    normalize_embeddings=True
)

hard_negative_embeddings = intent_model.encode(
    HARD_NEGATIVE_EXAMPLES,
    convert_to_tensor=True,
    normalize_embeddings=True
)



In [129]:
def count_keyword_hits(question: str, keywords: list) -> int:
    q = normalize_text(question)
    hits = 0
    for kw in keywords:
        if kw in q:
            hits += 1
    return hits


def _entity_match_score(question: str, entity_value: str) -> float:
    q_stems = set(stem_tokenize(question))
    e_stems = set(stem_tokenize(entity_value))

    if not q_stems or not e_stems:
        return 0.0

    overlap = len(q_stems & e_stems) / len(e_stems)
    char_score = SequenceMatcher(None, normalize_match_text(question), normalize_match_text(entity_value)).ratio()
    return 0.7 * overlap + 0.3 * char_score


def find_entity_hits(question: str, catalog: dict = None) -> list:
    if catalog is None:
        catalog = ENTITY_CATALOG

    q_norm = clean_entity_value(question)
    hits = []

    for entity_type, values in catalog.items():
        for value in values:
            if len(value) < 3:
                continue

            # 1) точное / подстрочное совпадение
            if value in q_norm:
                hits.append({
                    "entity_type": entity_type,
                    "entity_value": value,
                    "match_type": "exact",
                    "score": 1.0
                })
                continue

            # 2) мягкое совпадение по стемам
            score = _entity_match_score(q_norm, value)
            if score >= 0.62:
                hits.append({
                    "entity_type": entity_type,
                    "entity_value": value,
                    "match_type": "soft",
                    "score": round(float(score), 4)
                })

    dedup = []
    seen = set()
    for item in sorted(hits, key=lambda x: x["score"], reverse=True):
        key = (item["entity_type"], item["entity_value"])
        if key not in seen:
            seen.add(key)
            dedup.append(item)

    return dedup


def get_best_program_hint_score(question: str) -> float:
    q_norm = clean_entity_value(question)
    best = 0.0
    for value in ENTITY_CATALOG.get("program", []):
        score = _entity_match_score(q_norm, value)
        if score > best:
            best = score
    return round(float(best), 4)


def detect_table_rule(question: str) -> dict:
    """
    Возвращает:
    - is_table: вопрос похож на табличный запрос по программе
    - is_supported: поддерживаемый ли это атрибут таблицы
    - field_name: конкретное поле, если удалось понять
    - unsupported_label: если запрос по программе есть, но поля в данных нет
    """
    q_norm = clean_entity_value(question)

    # 1) неподдерживаемые, но тематические атрибуты
    for label, patterns in UNSUPPORTED_PROGRAM_FIELD_RULES.items():
        for pat in patterns:
            if pat in q_norm:
                return {
                    "is_table": True,
                    "is_supported": False,
                    "field_name": None,
                    "matched_keyword": pat,
                    "unsupported_label": label
                }

    # 2) явные поддерживаемые поля
    matches = []
    for rule in EXACT_FIELD_RULES:
        for pat in rule["patterns"]:
            if clean_entity_value(pat) in q_norm:
                matches.append({
                    "field_name": rule["field_name"],
                    "matched_keyword": pat,
                    "priority": len(pat)
                })

    # 3) отдельная развилка для ЕГЭ, если указано бюджет/платное более общими словами
    if ("егэ" in q_norm or "экзамен" in q_norm or "предмет" in q_norm or "вступительн" in q_norm):
        if "платн" in q_norm or "контракт" in q_norm:
            matches.append({"field_name": "eges_contract", "matched_keyword": "eges_contract", "priority": 100})
        elif "бюдж" in q_norm:
            matches.append({"field_name": "eges_budget", "matched_keyword": "eges_budget", "priority": 100})

    if not matches:
        return {
            "is_table": False,
            "is_supported": None,
            "field_name": None,
            "matched_keyword": None,
            "unsupported_label": None
        }

    matches = sorted(matches, key=lambda x: x["priority"], reverse=True)
    best = matches[0]

    return {
        "is_table": True,
        "is_supported": True,
        "field_name": best["field_name"],
        "matched_keyword": best["matched_keyword"],
        "unsupported_label": None
    }

In [130]:
def is_in_admission_domain(
    question: str,
    semantic_threshold: float = 0.50,
    margin_threshold: float = 0.02
) -> dict:
    """
    Более мягкая, но всё ещё безопасная проверка домена.
    Идея: лучше пропустить тематический вопрос внутрь системы
    и честно вернуть "не найдено", чем слишком рано отрезать его как out_of_scope.
    """
    q_norm = normalize_text(question)

    q_emb = intent_model.encode(
        q_norm,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    domain_sims = util.cos_sim(q_emb, domain_example_embeddings)[0].cpu().numpy()
    domain_best_idx = int(np.argmax(domain_sims))
    domain_score = float(np.max(domain_sims))

    negative_sims = util.cos_sim(q_emb, hard_negative_embeddings)[0].cpu().numpy()
    negative_best_idx = int(np.argmax(negative_sims))
    negative_score = float(np.max(negative_sims))

    strong_hits = count_keyword_hits(q_norm, DOMAIN_STRONG_KEYWORDS)
    weak_hits = count_keyword_hits(q_norm, DOMAIN_WEAK_KEYWORDS)
    entity_hits = find_entity_hits(q_norm)
    entity_hit_count = len(entity_hits)
    table_rule = detect_table_rule(q_norm)
    program_hint_score = get_best_program_hint_score(q_norm)

    margin = domain_score - negative_score

    if entity_hit_count >= 1:
        in_domain = True
        domain_reason = "entity_hit"

    elif program_hint_score >= 0.68:
        in_domain = True
        domain_reason = "program_hint"

    elif table_rule["is_table"]:
        in_domain = True
        domain_reason = "table_like_question"

    elif strong_hits >= 1 and domain_score >= semantic_threshold and margin >= margin_threshold:
        in_domain = True
        domain_reason = "strong_keyword_plus_semantics"

    elif weak_hits >= 2 and domain_score >= 0.53 and margin >= 0.01:
        in_domain = True
        domain_reason = "weak_keywords_plus_semantics"

    elif domain_score >= 0.64 and margin >= 0.04:
        in_domain = True
        domain_reason = "high_semantic_match"

    else:
        in_domain = False
        domain_reason = "outside_domain"

    return {
        "in_domain": in_domain,
        "domain_score": round(domain_score, 4),
        "negative_score": round(negative_score, 4),
        "domain_margin": round(margin, 4),
        "domain_strong_hits": strong_hits,
        "domain_weak_hits": weak_hits,
        "entity_hit_count": entity_hit_count,
        "entity_hits": entity_hits,
        "program_hint_score": program_hint_score,
        "matched_domain_example": DOMAIN_EXAMPLES[domain_best_idx],
        "matched_negative_example": HARD_NEGATIVE_EXAMPLES[negative_best_idx],
        "table_rule": table_rule,
        "domain_reason": domain_reason
    }

In [131]:
def classify_question_type_model(question: str, threshold: float = 0.44) -> dict:
    """
    Классификация внутри домена:
    - table
    - faq
    - general

    Логика:
    1) поддерживаемые и неподдерживаемые атрибуты программы считаем tabular-like;
    2) потом semantic intent;
    3) при сомнении между faq/general безопаснее выбирать general, а не out_of_scope.
    """
    table_rule = detect_table_rule(question)
    entity_hits = find_entity_hits(question)
    program_hint_score = get_best_program_hint_score(question)
    q_norm = normalize_text(question)

    if table_rule["is_table"] and (
        len(entity_hits) >= 1
        or program_hint_score >= 0.55
        or "программа" in q_norm
    ):
        return {
            "question_type": "table",
            "best_score": 1.0,
            "matched_example": f'rule_table::{table_rule.get("field_name")}::{table_rule.get("matched_keyword")}'
        }

    question_embedding = intent_model.encode(
        q_norm,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    scores = util.cos_sim(question_embedding, intent_example_embeddings)[0].cpu().numpy()

    label_bucket = {}
    for score, label, example in zip(scores, intent_label_names, intent_label_texts):
        label_bucket.setdefault(label, []).append((float(score), example))

    label_scores = []
    for label, pairs in label_bucket.items():
        pairs = sorted(pairs, key=lambda x: x[0], reverse=True)
        top_scores = [p[0] for p in pairs[:5]]
        agg_score = 0.7 * top_scores[0] + 0.3 * float(np.mean(top_scores))
        label_scores.append({
            "label": label,
            "score": agg_score,
            "matched_example": pairs[0][1]
        })

    label_scores = sorted(label_scores, key=lambda x: x["score"], reverse=True)
    best = label_scores[0]

    predicted_label = best["label"]
    best_score = float(best["score"])
    matched_example = best["matched_example"]

    if best_score < threshold:
        predicted_label = "general"

    if table_rule["is_table"] and predicted_label != "table" and program_hint_score >= 0.55:
        predicted_label = "table"
        matched_example = f'rule_table::{table_rule.get("field_name")}::{table_rule.get("matched_keyword")}'
        best_score = max(best_score, 0.75)

    return {
        "question_type": predicted_label,
        "best_score": round(best_score, 4),
        "matched_example": matched_example
    }


def classify_question_pipeline(question: str,
                               blocked_words: set = None,
                               intent_threshold: float = 0.44,
                               domain_threshold: float = 0.50) -> dict:
    q_norm = normalize_text(question)

    if blocked_words is not None:
        has_bad_words, found_words = contains_bad_words(q_norm, blocked_words)
        if has_bad_words:
            return {
                "question_type": "blocked",
                "best_score": None,
                "matched_example": None,
                "domain_score": None,
                "negative_score": None,
                "domain_margin": None,
                "domain_strong_hits": None,
                "domain_weak_hits": None,
                "entity_hit_count": None,
                "matched_domain_example": None,
                "matched_negative_example": None,
                "domain_reason": "blocked",
                "found_bad_words": found_words
            }

    domain_result = is_in_admission_domain(
        question=question,
        semantic_threshold=domain_threshold
    )

    if not domain_result["in_domain"]:
        return {
            "question_type": "out_of_scope",
            "best_score": None,
            "matched_example": None,
            "domain_score": domain_result["domain_score"],
            "negative_score": domain_result["negative_score"],
            "domain_margin": domain_result["domain_margin"],
            "domain_strong_hits": domain_result["domain_strong_hits"],
            "domain_weak_hits": domain_result["domain_weak_hits"],
            "entity_hit_count": domain_result["entity_hit_count"],
            "matched_domain_example": domain_result["matched_domain_example"],
            "matched_negative_example": domain_result["matched_negative_example"],
            "domain_reason": domain_result["domain_reason"],
            "found_bad_words": []
        }

    intent_result = classify_question_type_model(
        question=question,
        threshold=intent_threshold
    )

    intent_result["domain_score"] = domain_result["domain_score"]
    intent_result["negative_score"] = domain_result["negative_score"]
    intent_result["domain_margin"] = domain_result["domain_margin"]
    intent_result["domain_strong_hits"] = domain_result["domain_strong_hits"]
    intent_result["domain_weak_hits"] = domain_result["domain_weak_hits"]
    intent_result["entity_hit_count"] = domain_result["entity_hit_count"]
    intent_result["matched_domain_example"] = domain_result["matched_domain_example"]
    intent_result["matched_negative_example"] = domain_result["matched_negative_example"]
    intent_result["domain_reason"] = domain_result["domain_reason"]
    intent_result["found_bad_words"] = []

    return intent_result

In [132]:
def run_question_type_classification_model(df: pd.DataFrame,
                                           blocked_words: set,
                                           intent_threshold: float = 0.44,
                                           domain_threshold: float = 0.50) -> pd.DataFrame:
    df = df.copy()

    question_types = []
    best_scores = []
    matched_examples = []
    domain_scores = []
    negative_scores = []
    domain_margins = []
    domain_strong_hits_list = []
    domain_weak_hits_list = []
    entity_hit_count_list = []
    matched_domain_examples = []
    matched_negative_examples = []
    domain_reasons = []
    found_bad_words_list = []

    for question in df["question"]:
        result = classify_question_pipeline(
            question=question,
            blocked_words=blocked_words,
            intent_threshold=intent_threshold,
            domain_threshold=domain_threshold
        )

        question_types.append(result.get("question_type"))
        best_scores.append(result.get("best_score"))
        matched_examples.append(result.get("matched_example"))
        domain_scores.append(result.get("domain_score"))
        negative_scores.append(result.get("negative_score"))
        domain_margins.append(result.get("domain_margin"))
        domain_strong_hits_list.append(result.get("domain_strong_hits"))
        domain_weak_hits_list.append(result.get("domain_weak_hits"))
        entity_hit_count_list.append(result.get("entity_hit_count"))
        matched_domain_examples.append(result.get("matched_domain_example"))
        matched_negative_examples.append(result.get("matched_negative_example"))
        domain_reasons.append(result.get("domain_reason"))
        found_bad_words_list.append(", ".join(result.get("found_bad_words", [])))

    df["question_type"] = question_types
    df["best_score"] = best_scores
    df["matched_example"] = matched_examples
    df["domain_score"] = domain_scores
    df["negative_score"] = negative_scores
    df["domain_margin"] = domain_margins
    df["domain_strong_hits"] = domain_strong_hits_list
    df["domain_weak_hits"] = domain_weak_hits_list
    df["entity_hit_count"] = entity_hit_count_list
    df["matched_domain_example"] = matched_domain_examples
    df["matched_negative_example"] = matched_negative_examples
    df["domain_reason"] = domain_reasons
    df["found_bad_words_classifier"] = found_bad_words_list

    return df



In [133]:
abusive_words = load_words_from_txt(ABUSIVE_WORDS_FILE)
curse_words = load_words_from_txt(CURSE_WORDS_FILE)
all_bad_words = abusive_words.union(curse_words)

classified_df = run_question_type_classification_model(
    toxicity_df,
    blocked_words=all_bad_words,
    intent_threshold=0.44,
    domain_threshold=0.50
)

print(
    classified_df[
        [
            "question",
            "question_type",
            "domain_score",
            "negative_score",
            "domain_margin",
            "domain_strong_hits",
            "entity_hit_count",
            "domain_reason",
            "matched_example"
        ]
    ].head(20)
)



                                             question question_type  \
0   Какие ЕГЭ нужны для поступления на программу «...         table   
1   А на платное на «Анализ данных и искусственный...         table   
2       Сколько стоит обучение на бизнес-информатике?         table   
3   Какой проходной балл был в 2024 году на бизнес...         table   
4   Сколько бюджетных мест в 2025 году на программ...         table   
5             Сколько платных мест на веб-разработке?         table   
6       Есть ли бюджет на программе «Веб-разработка»?         table   
7   Сколько лет учиться на программе «Внешнеэконом...         table   
8   К какому мегакластеру относится программа «Мар...         table   
9   Какие требования по ЕГЭ на платное у программы...         table   
10  На сколько направлений одновременно можно пода...           faq   
11     Сколько лет действует олимпиада для права БВИ?           faq   
12  За что вообще начисляют дополнительные баллы п...           faq   
13    

# Маршрутизация
После того как система поняла тип вопроса, она выбирает нужный источник.  
То есть:   
•	если вопрос про стоимость / баллы / ЕГЭ / места → идём в таблицу all_program.xlsx;     
•	если вопрос типовой → ищем в FAQ Database.xlsx;    
•	если вопрос общий → ищем в текстовой базе знаний Database-2.xlsx.    


In [134]:
faq_df = pd.read_excel(FAQ_FILE)
general_df = pd.read_excel(GENERAL_FILE)
program_df = pd.read_excel(PROGRAM_FILE)

In [135]:
def route_question(question_type: str) -> dict:
    """
    Определяет, в какой источник отправлять вопрос дальше.
    """
    if question_type == "table":
        return {
            "route_target": "program_table",
            "source_name": "all_program.xlsx",
            "ready_for_retrieval": True
        }

    if question_type == "faq":
        return {
            "route_target": "faq_base",
            "source_name": "Database.xlsx",
            "ready_for_retrieval": True
        }

    if question_type == "general":
        return {
            "route_target": "general_base",
            "source_name": "Database-2.xlsx",
            "ready_for_retrieval": True
        }

    if question_type == "blocked":
        return {
            "route_target": "blocked",
            "source_name": None,
            "ready_for_retrieval": False
        }

    if question_type == "out_of_scope":
        return {
            "route_target": "out_of_scope",
            "source_name": None,
            "ready_for_retrieval": False
        }

    return {
        "route_target": "unknown",
        "source_name": None,
        "ready_for_retrieval": False
    }

In [136]:
def run_routing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    route_targets = []
    source_names = []
    ready_flags = []

    for _, row in df.iterrows():
        question_type = row["question_type"]

        result = route_question(question_type)

        route_targets.append(result["route_target"])
        source_names.append(result["source_name"])
        ready_flags.append(result["ready_for_retrieval"])

    df["route_target"] = route_targets
    df["source_name"] = source_names
    df["ready_for_retrieval"] = ready_flags

    return df

In [137]:
routed_df = run_routing(classified_df)

print(
    routed_df[
        [
            "question",
            "question_type",
            "route_target",
            "source_name",
            "ready_for_retrieval"
        ]
    ]
)

                                             question question_type  \
0   Какие ЕГЭ нужны для поступления на программу «...         table   
1   А на платное на «Анализ данных и искусственный...         table   
2       Сколько стоит обучение на бизнес-информатике?         table   
3   Какой проходной балл был в 2024 году на бизнес...         table   
4   Сколько бюджетных мест в 2025 году на программ...         table   
5             Сколько платных мест на веб-разработке?         table   
6       Есть ли бюджет на программе «Веб-разработка»?         table   
7   Сколько лет учиться на программе «Внешнеэконом...         table   
8   К какому мегакластеру относится программа «Мар...         table   
9   Какие требования по ЕГЭ на платное у программы...         table   
10  На сколько направлений одновременно можно пода...           faq   
11     Сколько лет действует олимпиада для права БВИ?           faq   
12  За что вообще начисляют дополнительные баллы п...           faq   
13    

# Подготовка запроса

In [138]:
# =========================
# 5. БЛОК ПОДГОТОВКИ ЗАПРОСА (candidate-first версия)
# =========================

from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import re

# -------------------------
# 5.1. Нормализация
# -------------------------

def normalize_for_match(text: str) -> str:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"[-–—/]", " ", text)
    text = re.sub(r"[^а-яa-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# -------------------------
# 5.2. Каталог программ
# -------------------------

program_df = pd.read_excel(PROGRAM_FILE).copy()
program_df["program"] = program_df["program"].fillna("").astype(str).str.strip()
program_df = program_df[program_df["program"] != ""].copy()

unique_programs = program_df["program"].drop_duplicates().tolist()
program_match_texts = [normalize_for_match(p) for p in unique_programs]

program_embeddings = intent_model.encode(
    program_match_texts,
    convert_to_tensor=True,
    normalize_embeddings=True
)


# -------------------------
# 5.3. Группы полей
# -------------------------

FIELD_GROUPS = [
    {
        "group_name": "cost",
        "label": "стоимость обучения",
        "prototypes": [
            "стоимость обучения",
            "сколько стоит обучение",
            "цена программы",
            "стоимость учебы"
        ]
    },
    {
        "group_name": "pass",
        "label": "проходной балл",
        "prototypes": [
            "проходной балл",
            "баллы для поступления",
            "какой проходной балл",
            "минимальный проходной балл"
        ]
    },
    {
        "group_name": "places",
        "label": "места",
        "prototypes": [
            "сколько мест на программе",
            "количество мест",
            "места на обучение",
            "набор на программу"
        ]
    },
    {
        "group_name": "eges",
        "label": "ЕГЭ",
        "prototypes": [
            "какие егэ нужны",
            "какие экзамены нужны",
            "предметы егэ для поступления",
            "вступительные испытания"
        ]
    },
    {
        "group_name": "edu_form",
        "label": "форма обучения",
        "prototypes": [
            "форма обучения",
            "очная или заочная форма",
            "формат обучения"
        ]
    },
    {
        "group_name": "edu_years",
        "label": "длительность обучения",
        "prototypes": [
            "сколько лет учиться",
            "длительность обучения",
            "срок обучения"
        ]
    },
    {
        "group_name": "tracks",
        "label": "треки",
        "prototypes": [
            "какие есть треки",
            "профили программы",
            "специализации программы",
            "образовательные треки"
        ]
    },
    {
        "group_name": "institute",
        "label": "институт",
        "prototypes": [
            "какой институт",
            "какая школа",
            "где реализуется программа"
        ]
    },
    {
        "group_name": "megacluster",
        "label": "мегакластер",
        "prototypes": [
            "какой мегакластер",
            "к какому мегакластеру относится",
            "мегакластер программы"
        ]
    },
    {
        "group_name": "major",
        "label": "направление подготовки",
        "prototypes": [
            "направление подготовки",
            "код направления",
            "какое направление подготовки"
        ]
    },
    {
        "group_name": "desc",
        "label": "описание программы",
        "prototypes": [
            "о чем программа",
            "описание программы",
            "что изучают на программе",
            "в чем суть программы"
        ]
    },
    {
        "group_name": "skills",
        "label": "навыки выпускника",
        "prototypes": [
            "чему научусь",
            "какие навыки получу",
            "компетенции выпускника",
            "какие навыки будут"
        ]
    }
]


# -------------------------
# 5.4. Поля внутри групп
# -------------------------

FIELD_OPTIONS = [
    {
        "field_name": "cost",
        "group_name": "cost",
        "label": "стоимость обучения",
        "prototypes": [
            "стоимость обучения",
            "сколько стоит обучение",
            "цена программы"
        ]
    },
    {
        "field_name": "pass_2024",
        "group_name": "pass",
        "label": "проходной балл",
        "prototypes": [
            "проходной балл",
            "баллы для поступления",
            "минимальный проходной балл"
        ]
    },
    {
        "field_name": "budget_2025",
        "group_name": "places",
        "label": "бюджетные места",
        "prototypes": [
            "бюджетные места",
            "места на бюджете",
            "количество бюджетных мест"
        ]
    },
    {
        "field_name": "contract_2025",
        "group_name": "places",
        "label": "платные места",
        "prototypes": [
            "платные места",
            "контрактные места",
            "места на платном обучении"
        ]
    },
    {
        "field_name": "eges_budget",
        "group_name": "eges",
        "label": "ЕГЭ для бюджета",
        "prototypes": [
            "егэ для бюджета",
            "какие егэ нужны на бюджет",
            "экзамены для поступления на бюджет"
        ]
    },
    {
        "field_name": "eges_contract",
        "group_name": "eges",
        "label": "ЕГЭ для платного обучения",
        "prototypes": [
            "егэ для платного обучения",
            "какие егэ нужны на платное",
            "экзамены для поступления на контракт"
        ]
    },
    {
        "field_name": "edu_form",
        "group_name": "edu_form",
        "label": "форма обучения",
        "prototypes": [
            "форма обучения",
            "очная форма",
            "заочная форма",
            "формат обучения"
        ]
    },
    {
        "field_name": "edu_years",
        "group_name": "edu_years",
        "label": "длительность обучения",
        "prototypes": [
            "сколько лет учиться",
            "длительность обучения",
            "срок обучения"
        ]
    },
    {
        "field_name": "tracks",
        "group_name": "tracks",
        "label": "треки",
        "prototypes": [
            "какие есть треки",
            "профили программы",
            "специализации программы"
        ]
    },
    {
        "field_name": "institute",
        "group_name": "institute",
        "label": "институт",
        "prototypes": [
            "какой институт",
            "какая школа",
            "где реализуется программа"
        ]
    },
    {
        "field_name": "megacluster",
        "group_name": "megacluster",
        "label": "мегакластер",
        "prototypes": [
            "какой мегакластер",
            "к какому мегакластеру относится",
            "мегакластер программы"
        ]
    },
    {
        "field_name": "major",
        "group_name": "major",
        "label": "направление подготовки",
        "prototypes": [
            "направление подготовки",
            "код направления",
            "какое направление подготовки"
        ]
    },
    {
        "field_name": "desc",
        "group_name": "desc",
        "label": "описание программы",
        "prototypes": [
            "о чем программа",
            "описание программы",
            "что изучают на программе"
        ]
    },
    {
        "field_name": "skills",
        "group_name": "skills",
        "label": "навыки выпускника",
        "prototypes": [
            "чему научусь",
            "какие навыки получу",
            "компетенции выпускника"
        ]
    }
]

FIELD_LABELS = {x["field_name"]: x["label"] for x in FIELD_OPTIONS}


# -------------------------
# 5.5. Эмбеддинги групп и полей
# -------------------------

group_proto_texts = []
group_proto_meta = []

for group in FIELD_GROUPS:
    for proto in group["prototypes"]:
        group_proto_texts.append(normalize_for_match(proto))
        group_proto_meta.append({
            "group_name": group["group_name"],
            "group_label": group["label"],
            "prototype": proto
        })

group_proto_embeddings = intent_model.encode(
    group_proto_texts,
    convert_to_tensor=True,
    normalize_embeddings=True
)

field_proto_texts = []
field_proto_meta = []

for field in FIELD_OPTIONS:
    for proto in field["prototypes"]:
        field_proto_texts.append(normalize_for_match(proto))
        field_proto_meta.append({
            "field_name": field["field_name"],
            "group_name": field["group_name"],
            "label": field["label"],
            "prototype": proto
        })

field_proto_embeddings = intent_model.encode(
    field_proto_texts,
    convert_to_tensor=True,
    normalize_embeddings=True
)

In [139]:
def lexical_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def embed_text(text: str):
    return intent_model.encode(
        normalize_for_match(text),
        convert_to_tensor=True,
        normalize_embeddings=True
    )


def aggregate_scores_by_key(similarities, meta_list, key_name, label_name=None):
    """
    Собирает несколько prototype-score в один score по группе или полю.
    """
    bucket = {}

    for score, meta in zip(similarities, meta_list):
        key = meta[key_name]
        if key not in bucket:
            bucket[key] = {
                "key": key,
                "scores": [],
                "label": meta[label_name] if label_name else key
            }
        bucket[key]["scores"].append(float(score))

    result = []
    for key, info in bucket.items():
        max_score = max(info["scores"])
        mean_top2 = np.mean(sorted(info["scores"], reverse=True)[:2])
        final_score = 0.7 * max_score + 0.3 * mean_top2

        result.append({
            "key": key,
            "label": info["label"],
            "max_score": round(float(max_score), 4),
            "final_score": round(float(final_score), 4)
        })

    result = sorted(result, key=lambda x: x["final_score"], reverse=True)
    return result


def remove_program_from_question(question: str, program_candidate: str) -> str:
    """
    Удаляем лучший кандидат программы из вопроса,
    чтобы поле определялось по 'остатку' вопроса.
    """
    q = normalize_for_match(question)

    if not program_candidate:
        return q

    p = normalize_for_match(program_candidate)
    q_removed = re.sub(rf"\b{re.escape(p)}\b", " ", q)
    q_removed = re.sub(r"\s+", " ", q_removed).strip()

    return q_removed if q_removed else q

In [140]:
def _extract_quoted_fragments(text: str) -> list:
    text = str(text)
    fragments = []
    fragments.extend(re.findall(r"[«\"]([^»\"]+)[»\"]", text))
    fragments.extend(re.findall(r"'([^']+)'", text))
    return [normalize_for_match(x) for x in fragments if str(x).strip()]


def get_top_program_candidates(question: str, top_k: int = 5) -> list:
    q_norm = normalize_for_match(question)
    quoted_fragments = _extract_quoted_fragments(question)

    # Если в вопросе есть название в кавычках, сначала матчим именно его:
    query_variants = quoted_fragments if quoted_fragments else [q_norm]

    bucket = {}
    for query_text in query_variants:
        q_emb = embed_text(query_text)

        sem_scores = util.cos_sim(q_emb, program_embeddings)[0].cpu().numpy()
        lex_scores = np.array([
            lexical_similarity(query_text, p_norm) for p_norm in program_match_texts
        ])

        final_scores = 0.78 * sem_scores + 0.22 * lex_scores
        top_idx = np.argsort(final_scores)[::-1][:top_k]

        for idx in top_idx:
            program_name = unique_programs[idx]
            cand = {
                "program": program_name,
                "score": round(float(final_scores[idx]), 4),
                "semantic_score": round(float(sem_scores[idx]), 4),
                "lexical_score": round(float(lex_scores[idx]), 4)
            }
            if program_name not in bucket or cand["score"] > bucket[program_name]["score"]:
                bucket[program_name] = cand

    candidates = sorted(bucket.values(), key=lambda x: x["score"], reverse=True)
    return candidates[:top_k]


def detect_program_candidate(question: str,
                             min_score: float = 0.40,
                             ambiguity_gap: float = 0.04) -> dict:
    candidates = get_top_program_candidates(question, top_k=5)

    if not candidates:
        return {
            "program_candidate": None,
            "program_match_score": None,
            "program_candidates_top3": [],
            "program_found": False,
            "program_needs_clarification": True
        }

    best = candidates[0]
    second = candidates[1] if len(candidates) > 1 else None

    program_found = best["score"] >= min_score
    program_needs_clarification = not program_found

    if second is not None and abs(best["score"] - second["score"]) < ambiguity_gap:
        program_needs_clarification = True

    return {
        "program_candidate": best["program"] if program_found else None,
        "program_match_score": best["score"],
        "program_candidates_top3": candidates[:3],
        "program_found": program_found,
        "program_needs_clarification": program_needs_clarification
    }

In [141]:
def detect_field_group_candidates(question: str, top_k: int = 5) -> list:
    q_emb = embed_text(question)
    sims = util.cos_sim(q_emb, group_proto_embeddings)[0].cpu().numpy()

    ranked_groups = aggregate_scores_by_key(
        similarities=sims,
        meta_list=group_proto_meta,
        key_name="group_name",
        label_name="group_label"
    )
    return ranked_groups[:top_k]


def detect_best_field_group(question: str,
                            min_score: float = 0.30,
                            ambiguity_gap: float = 0.05) -> dict:
    ranked_groups = detect_field_group_candidates(question, top_k=5)

    if not ranked_groups:
        return {
            "field_group": None,
            "field_group_label": None,
            "field_group_score": None,
            "field_group_candidates_top3": [],
            "group_found": False,
            "group_needs_clarification": True
        }

    best = ranked_groups[0]
    second = ranked_groups[1] if len(ranked_groups) > 1 else None

    group_found = best["final_score"] >= min_score
    group_needs_clarification = not group_found

    if second is not None and abs(best["final_score"] - second["final_score"]) < ambiguity_gap:
        group_needs_clarification = True

    return {
        "field_group": best["key"] if group_found else None,
        "field_group_label": best["label"] if group_found else None,
        "field_group_score": best["final_score"],
        "field_group_candidates_top3": ranked_groups[:3],
        "group_found": group_found,
        "group_needs_clarification": group_needs_clarification
    }

In [142]:
def detect_field_candidates_within_group(question: str,
                                         target_group: str,
                                         top_k: int = 3) -> list:
    q_emb = embed_text(question)
    sims = util.cos_sim(q_emb, field_proto_embeddings)[0].cpu().numpy()

    filtered_sims = []
    filtered_meta = []

    for score, meta in zip(sims, field_proto_meta):
        if meta["group_name"] == target_group:
            filtered_sims.append(score)
            filtered_meta.append(meta)

    ranked_fields = aggregate_scores_by_key(
        similarities=filtered_sims,
        meta_list=filtered_meta,
        key_name="field_name",
        label_name="label"
    )
    return ranked_fields[:top_k]


def merge_rankings(rank_a: list, rank_b: list, weight_a: float = 0.55, weight_b: float = 0.45) -> list:
    """
    Объединяем ranking по полному вопросу и по question_without_program.
    """
    bucket = {}

    for item in rank_a:
        bucket[item["key"]] = {
            "key": item["key"],
            "label": item["label"],
            "score_a": item["final_score"],
            "score_b": 0.0
        }

    for item in rank_b:
        if item["key"] not in bucket:
            bucket[item["key"]] = {
                "key": item["key"],
                "label": item["label"],
                "score_a": 0.0,
                "score_b": item["final_score"]
            }
        else:
            bucket[item["key"]]["score_b"] = item["final_score"]

    merged = []
    for _, item in bucket.items():
        final_score = weight_a * item["score_a"] + weight_b * item["score_b"]
        merged.append({
            "key": item["key"],
            "label": item["label"],
            "score_full_question": round(float(item["score_a"]), 4),
            "score_without_program": round(float(item["score_b"]), 4),
            "final_score": round(float(final_score), 4)
        })

    merged = sorted(merged, key=lambda x: x["final_score"], reverse=True)
    return merged

In [143]:
EXACT_FIELD_TO_GROUP = {
    "cost": ("cost", "стоимость обучения"),
    "pass_2024": ("pass", "проходной балл"),
    "budget_2025": ("places", "места"),
    "contract_2025": ("places", "места"),
    "eges_budget": ("eges", "ЕГЭ"),
    "eges_contract": ("eges", "ЕГЭ"),
    "edu_form": ("edu_form", "форма обучения"),
    "edu_years": ("edu_years", "длительность обучения"),
    "tracks": ("tracks", "треки"),
    "institute": ("institute", "институт"),
    "megacluster": ("megacluster", "мегакластер"),
    "major": ("major", "направление подготовки"),
    "desc": ("desc", "описание программы"),
    "skills": ("skills", "навыки выпускника")
}

def build_retrieval_query(route_target: str,
                          normalized_question: str,
                          program_candidate: str = None,
                          field_group_label: str = None,
                          field_candidates_topk: list = None) -> str:
    parts = []

    if route_target == "program_table":
        if program_candidate:
            parts.append(program_candidate)

        if field_group_label:
            parts.append(field_group_label)

        if field_candidates_topk:
            candidate_labels = [x["label"] for x in field_candidates_topk[:2]]
            if candidate_labels:
                parts.append(" / ".join(candidate_labels))

        parts.append(normalized_question)
        return " | ".join([p for p in parts if p]).strip()

    return normalized_question


def prepare_single_query(question: str,
                         route_target: str,
                         ready_for_retrieval: bool) -> dict:
    normalized_question = normalize_for_match(question)
    table_rule = detect_table_rule(question)

    result = {
        "normalized_question": normalized_question,

        "program_candidate": None,
        "program_match_score": None,
        "program_candidates_top3": None,
        "program_found": False,
        "program_needs_clarification": False,

        "question_without_program": None,

        "field_group": None,
        "field_group_label": None,
        "field_group_score": None,
        "field_group_candidates_top3": None,
        "group_found": False,
        "group_needs_clarification": False,

        "field_candidates_topk": None,
        "requested_field_best": None,
        "requested_field_best_label": None,
        "requested_field_best_score": None,

        "retrieval_query": normalized_question,
        "needs_clarification": False,
        "clarification_comment": None,

        "table_rule_is_table": table_rule["is_table"],
        "table_rule_is_supported": table_rule["is_supported"],
        "unsupported_field_label": table_rule.get("unsupported_label")
    }

    if not ready_for_retrieval:
        result["clarification_comment"] = "Вопрос не готов к retrieval."
        return result

    if route_target in {"faq_base", "general_base"}:
        result["retrieval_query"] = build_retrieval_query(
            route_target=route_target,
            normalized_question=normalized_question
        )
        return result

    if route_target == "program_table":
        program_info = detect_program_candidate(question)
        result.update(program_info)

        question_without_program = remove_program_from_question(
            question=question,
            program_candidate=result["program_candidate"]
        )
        result["question_without_program"] = question_without_program

        # 1) если точное поле понято правилами — используем его напрямую
        if table_rule["is_table"] and table_rule["is_supported"] and table_rule.get("field_name") in EXACT_FIELD_TO_GROUP:
            exact_field = table_rule["field_name"]
            group_name, group_label = EXACT_FIELD_TO_GROUP[exact_field]

            result["field_group"] = group_name
            result["field_group_label"] = group_label
            result["field_group_score"] = 1.0
            result["field_group_candidates_top3"] = [{
                "key": group_name,
                "label": group_label,
                "max_score": 1.0,
                "final_score": 1.0
            }]
            result["group_found"] = True
            result["group_needs_clarification"] = False

            result["requested_field_best"] = exact_field
            result["requested_field_best_label"] = FIELD_LABELS.get(exact_field, exact_field)
            result["requested_field_best_score"] = 1.0
            result["field_candidates_topk"] = [{
                "key": exact_field,
                "label": FIELD_LABELS.get(exact_field, exact_field),
                "score_full_question": 1.0,
                "score_without_program": 1.0,
                "final_score": 1.0
            }]

        # 2) если запрос тематический, но поле в таблице не поддерживается —
        # не пытаемся угадать соседнее поле
        elif table_rule["is_table"] and table_rule["is_supported"] is False:
            result["field_group"] = None
            result["field_group_label"] = None
            result["field_group_score"] = None
            result["field_group_candidates_top3"] = []
            result["group_found"] = False
            result["group_needs_clarification"] = False
            result["requested_field_best"] = None
            result["requested_field_best_label"] = None
            result["requested_field_best_score"] = None
            result["field_candidates_topk"] = []

        # 3) иначе оставляем semantic fallback по старой логике
        else:
            group_info = detect_best_field_group(question_without_program)
            result.update(group_info)

            if result["group_found"] and result["field_group"] is not None:
                rank_full = detect_field_candidates_within_group(
                    question=normalized_question,
                    target_group=result["field_group"],
                    top_k=3
                )
                rank_wo_program = detect_field_candidates_within_group(
                    question=question_without_program,
                    target_group=result["field_group"],
                    top_k=3
                )
                merged_rank = merge_rankings(rank_full, rank_wo_program)

                result["field_candidates_topk"] = merged_rank[:3]

                if merged_rank:
                    best_field = merged_rank[0]
                    result["requested_field_best"] = best_field["key"]
                    result["requested_field_best_label"] = best_field["label"]
                    result["requested_field_best_score"] = best_field["final_score"]

        result["retrieval_query"] = build_retrieval_query(
            route_target=route_target,
            normalized_question=normalized_question,
            program_candidate=result["program_candidate"],
            field_group_label=result["field_group_label"],
            field_candidates_topk=result["field_candidates_topk"]
        )

        needs_clarification = (
            result["program_needs_clarification"]
            or (
                table_rule["is_table"]
                and table_rule["is_supported"] is True
                and result["group_needs_clarification"]
            )
        )
        result["needs_clarification"] = needs_clarification

        comments = []
        if result["program_needs_clarification"]:
            comments.append("не удалось уверенно определить программу")
        if table_rule["is_table"] and table_rule["is_supported"] is False:
            comments.append("запрошенного показателя нет в таблице")
        elif result["group_needs_clarification"]:
            comments.append("не удалось уверенно определить группу поля")

        if comments:
            result["clarification_comment"] = "; ".join(comments)

        return result

    result["clarification_comment"] = "Неизвестный route_target."
    return result

In [144]:
def run_query_preparation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    prepared_rows = []

    for _, row in df.iterrows():
        prepared = prepare_single_query(
            question=row["question"],
            route_target=row["route_target"],
            ready_for_retrieval=row["ready_for_retrieval"]
        )

        merged_row = row.to_dict()
        merged_row.update(prepared)
        prepared_rows.append(merged_row)

    return pd.DataFrame(prepared_rows)

In [145]:
prepared_df_v3 = run_query_preparation(routed_df)

prepared_df_v3[
    [
        "question",
        "route_target",
        "program_candidate",
        "question_without_program",
        "field_group",
        "field_group_label",
        "requested_field_best",
        "requested_field_best_label",
        "field_candidates_topk",
        "retrieval_query",
        "needs_clarification",
        "clarification_comment"
    ]
].head(20)

,question,route_target,program_candidate,question_without_program,field_group,field_group_label,requested_field_best,requested_field_best_label,field_candidates_topk,retrieval_query,needs_clarification,clarification_comment
0,Какие ЕГЭ нужны для поступления на программу «...,program_table,анализ данных и искусственный интеллект,какие егэ нужны для поступления на программу н...,eges,ЕГЭ,eges_budget,ЕГЭ для бюджета,"[{'key': 'eges_budget', 'label': 'ЕГЭ для бюдж...",анализ данных и искусственный интеллект | ЕГЭ ...,False,None
1,А на платное на «Анализ данных и искусственный...,program_table,анализ данных и искусственный интеллект,а на платное на какие минимальные егэ,eges,ЕГЭ,eges_contract,ЕГЭ для платного обучения,"[{'key': 'eges_contract', 'label': 'ЕГЭ для пл...",анализ данных и искусственный интеллект | ЕГЭ ...,False,None
2,Сколько стоит обучение на бизнес-информатике?,program_table,бизнес-информатика,сколько стоит обучение на бизнес информатике,cost,стоимость обучения,cost,стоимость обучения,"[{'key': 'cost', 'label': 'стоимость обучения'...",бизнес-информатика | стоимость обучения | стои...,False,None
3,Какой проходной балл был в 2024 году на бизнес...,program_table,бизнес-информатика,какой проходной балл был в 2024 году на бизнес...,pass,проходной балл,pass_2024,проходной балл,"[{'key': 'pass_2024', 'label': 'проходной балл...",бизнес-информатика | проходной балл | проходно...,False,None
4,Сколько бюджетных мест в 2025 году на программ...,program_table,внешняя политика и государственное управление,сколько бюджетных мест в 2025 году на программе,places,места,budget_2025,бюджетные места,"[{'key': 'budget_2025', 'label': 'бюджетные ме...",внешняя политика и государственное управление ...,False,None
5,Сколько платных мест на веб-разработке?,program_table,веб-разработка,сколько платных мест на веб разработке,places,места,contract_2025,платные места,"[{'key': 'contract_2025', 'label': 'платные ме...",веб-разработка | места | платные места / бюдже...,False,None
6,Есть ли бюджет на программе «Веб-разработка»?,program_table,веб-разработка,есть ли бюджет на программе,places,места,budget_2025,бюджетные места,"[{'key': 'budget_2025', 'label': 'бюджетные ме...",веб-разработка | места | бюджетные места | ест...,False,None
7,Сколько лет учиться на программе «Внешнеэконом...,program_table,внешнеэкономическая деятельность,сколько лет учиться на программе,edu_years,длительность обучения,edu_years,длительность обучения,"[{'key': 'edu_years', 'label': 'длительность о...",внешнеэкономическая деятельность | длительност...,False,None
8,К какому мегакластеру относится программа «Мар...,program_table,маркетинг,к какому мегакластеру относится программа,megacluster,мегакластер,megacluster,мегакластер,"[{'key': 'megacluster', 'label': 'мегакластер'...",маркетинг | мегакластер | мегакластер | к како...,False,None
9,Какие требования по ЕГЭ на платное у программы...,program_table,деловая журналистика,какие требования по егэ на платное у программы,eges,ЕГЭ,eges_contract,ЕГЭ для платного обучения,"[{'key': 'eges_contract', 'label': 'ЕГЭ для пл...",деловая журналистика | ЕГЭ | ЕГЭ для платного ...,False,None


# Retrieval внутри нужного источника

In [146]:
# =========================
# 6. БЛОК RETRIEVAL ВНУТРИ НУЖНОГО ИСТОЧНИКА
# =========================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- 6.1. Загрузка источников ----------
program_source_df = pd.read_excel(PROGRAM_FILE).copy()
faq_df = pd.read_excel(FAQ_FILE).copy()
general_df = pd.read_excel(GENERAL_FILE).copy()

program_source_df = program_source_df.fillna("")
faq_df = faq_df.fillna("")
general_df = general_df.fillna("")

for col in program_source_df.columns:
    program_source_df[col] = program_source_df[col].astype(str)

for col in faq_df.columns:
    faq_df[col] = faq_df[col].astype(str)

for col in general_df.columns:
    general_df[col] = general_df[col].astype(str)

# ---------- 6.2. Нормализованные поля ----------
program_source_df["program_norm"] = program_source_df["program"].apply(normalize_for_match)

faq_df["question_norm"] = faq_df["Question"].apply(normalize_for_match)
faq_df["answer_norm"] = faq_df["Answer"].apply(normalize_for_match)
faq_df["search_text"] = (faq_df["Question"] + " " + faq_df["Answer"]).apply(normalize_for_match)

general_df["header_norm"] = general_df["header"].apply(normalize_for_match)
general_df["text_norm"] = general_df["text"].apply(normalize_for_match)
general_df["search_text"] = (general_df["header"] + " " + general_df["text"]).apply(normalize_for_match)

# ---------- 6.3. Эмбеддинги ----------
faq_question_embeddings = intent_model.encode(
    faq_df["question_norm"].tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True
)

general_embeddings = intent_model.encode(
    general_df["search_text"].tolist(),
    convert_to_tensor=True,
    normalize_embeddings=True
)

# ---------- 6.4. TF-IDF для гибридного retrieval ----------
faq_vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1
)
faq_tfidf = faq_vectorizer.fit_transform(faq_df["search_text"].tolist())

general_vectorizer = TfidfVectorizer(
    lowercase=True,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1
)
general_tfidf = general_vectorizer.fit_transform(general_df["search_text"].tolist())

In [147]:
def get_out_of_scope_response() -> str:
    return (
        "Я помогаю с вопросами о поступлении: образовательные программы, "
        "стоимость обучения, проходные баллы, ЕГЭ, количество мест и мегакластеры. "
        "Если ваш вопрос связан с поступлением, сформулируйте его, пожалуйста, подробнее."
    )

In [148]:
def get_blocked_response() -> str:
    return (
        "Пожалуйста, общайтесь корректно. "
        "Я помогу с вопросами о поступлении, программах, стоимости обучения, "
        "ЕГЭ и других темах, если сформулировать запрос спокойно."
    )

In [149]:
def safe_to_list(x):
    if isinstance(x, list):
        return x
    return []


def token_overlap_ratio(a: str, b: str) -> float:
    a_set = set(stem_tokenize(a))
    b_set = set(stem_tokenize(b))
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)


def hybrid_row_score(query: str, candidate_text: str, semantic_score: float | None = None) -> float:
    lex = lexical_similarity(normalize_for_match(query), normalize_for_match(candidate_text))
    overlap = token_overlap_ratio(query, candidate_text)
    if semantic_score is None:
        return 0.55 * lex + 0.45 * overlap
    return 0.55 * float(semantic_score) + 0.25 * lex + 0.20 * overlap


def rank_program_rows(question: str, program_candidates_top3: list, top_k: int = 3) -> list:
    ranked_rows = []

    if not isinstance(program_candidates_top3, list) or len(program_candidates_top3) == 0:
        return ranked_rows

    q_norm = normalize_for_match(question)

    for cand in program_candidates_top3:
        program_name = cand.get("program")
        program_score = float(cand.get("score", 0.0))

        subset = program_source_df[program_source_df["program"] == program_name].copy()
        if subset.empty:
            continue

        row_scores = []
        for idx, row in subset.iterrows():
            row_text = " ".join([
                row.get("program", ""),
                row.get("megacluster", ""),
                row.get("institute", ""),
                row.get("major", ""),
                row.get("tracks", ""),
                row.get("desc", ""),
                row.get("skills", "")
            ])
            final_score = hybrid_row_score(
                query=q_norm,
                candidate_text=row_text,
                semantic_score=program_score
            )

            row_scores.append({
                "row_idx": idx,
                "program": row.get("program", ""),
                "row_score": round(float(final_score), 4),
                "program_candidate_score": round(float(program_score), 4)
            })

        row_scores = sorted(row_scores, key=lambda x: x["row_score"], reverse=True)
        ranked_rows.extend(row_scores[:top_k])

    dedup = {}
    for item in ranked_rows:
        idx = item["row_idx"]
        if idx not in dedup or item["row_score"] > dedup[idx]["row_score"]:
            dedup[idx] = item

    ranked_rows = sorted(dedup.values(), key=lambda x: x["row_score"], reverse=True)
    return ranked_rows[:top_k]


def get_candidate_field_names(row: pd.Series, requested_field_best, field_candidates_topk):
    candidates = []

    if requested_field_best and requested_field_best in row.index:
        candidates.append(requested_field_best)

    # Если точное поле уже найдено, соседние поля не подмешиваем
    if candidates:
        return candidates

    if isinstance(field_candidates_topk, list):
        for item in field_candidates_topk:
            field_name = item.get("key")
            if field_name in row.index and field_name not in candidates:
                candidates.append(field_name)

    return candidates


def retrieve_from_program_table(question: str,
                                program_candidate: str,
                                program_candidates_top3: list,
                                requested_field_best: str,
                                field_candidates_topk: list,
                                field_group: str,
                                unsupported_field_label: str = None) -> dict:
    ranked_rows = rank_program_rows(
        question=question,
        program_candidates_top3=program_candidates_top3,
        top_k=3
    )

    if len(ranked_rows) == 0:
        return {
            "retrieval_status": "program_not_found",
            "retrieval_source": "all_program.xlsx",
            "retrieval_type": "program_table",
            "matched_program": None,
            "matched_row_idx": None,
            "matched_field": None,
            "matched_value": None,
            "field_values_found": [],
            "retrieval_score": None,
            "unsupported_field_label": unsupported_field_label
        }

    best_row_info = ranked_rows[0]
    row = program_source_df.loc[best_row_info["row_idx"]]

    # Если поле тематическое, но не поддерживается в таблице — честно говорим об этом
    if unsupported_field_label:
        return {
            "retrieval_status": "field_not_found",
            "retrieval_source": "all_program.xlsx",
            "retrieval_type": "program_table",
            "matched_program": row.get("program", ""),
            "matched_row_idx": int(best_row_info["row_idx"]),
            "matched_field": None,
            "matched_value": None,
            "field_group": field_group,
            "field_values_found": [],
            "retrieval_score": best_row_info["row_score"],
            "unsupported_field_label": unsupported_field_label
        }

    candidate_fields = get_candidate_field_names(
        row=row,
        requested_field_best=requested_field_best,
        field_candidates_topk=field_candidates_topk
    )

    field_values_found = []
    for field_name in candidate_fields:
        value = row.get(field_name, "")
        if str(value).strip() != "":
            field_values_found.append({
                "field_name": field_name,
                "field_label": FIELD_LABELS.get(field_name, field_name),
                "value": value
            })

    matched_field = None
    matched_value = None

    if len(field_values_found) > 0:
        matched_field = field_values_found[0]["field_name"]
        matched_value = field_values_found[0]["value"]

    retrieval_status = "found" if matched_value is not None else "field_not_found"

    return {
        "retrieval_status": retrieval_status,
        "retrieval_source": "all_program.xlsx",
        "retrieval_type": "program_table",
        "matched_program": row.get("program", ""),
        "matched_row_idx": int(best_row_info["row_idx"]),
        "matched_field": matched_field,
        "matched_value": matched_value,
        "field_group": field_group,
        "field_values_found": field_values_found,
        "retrieval_score": best_row_info["row_score"],
        "unsupported_field_label": unsupported_field_label
    }


def retrieve_from_faq(question: str, top_k: int = 3) -> dict:
    q_norm = normalize_for_match(question)
    q_emb = embed_text(q_norm)

    sem_sims = util.cos_sim(q_emb, faq_question_embeddings)[0].cpu().numpy()
    tfidf_vec = faq_vectorizer.transform([q_norm])
    tfidf_sims = cosine_similarity(tfidf_vec, faq_tfidf)[0]

    scores = []
    for idx, row in faq_df.iterrows():
        score = 0.60 * float(sem_sims[idx]) + 0.25 * float(tfidf_sims[idx]) + 0.15 * token_overlap_ratio(q_norm, row["search_text"])
        scores.append(score)

    top_idx = np.argsort(scores)[::-1][:top_k]

    faq_hits = []
    for idx in top_idx:
        faq_hits.append({
            "faq_question": faq_df.iloc[idx]["Question"],
            "faq_answer": faq_df.iloc[idx]["Answer"],
            "faq_question_type": faq_df.iloc[idx].get("Question type", ""),
            "score": round(float(scores[idx]), 4)
        })

    if len(faq_hits) == 0:
        return {
            "retrieval_status": "not_found",
            "retrieval_source": "Database.xlsx",
            "retrieval_type": "faq_base",
            "matched_question": None,
            "matched_answer": None,
            "retrieval_score": None,
            "faq_hits_topk": []
        }

    best = faq_hits[0]

    if float(best["score"]) < 0.33:
        return {
            "retrieval_status": "not_found",
            "retrieval_source": "Database.xlsx",
            "retrieval_type": "faq_base",
            "matched_question": None,
            "matched_answer": None,
            "retrieval_score": best["score"],
            "faq_hits_topk": faq_hits
        }

    return {
        "retrieval_status": "found",
        "retrieval_source": "Database.xlsx",
        "retrieval_type": "faq_base",
        "matched_question": best["faq_question"],
        "matched_answer": best["faq_answer"],
        "matched_question_type": best["faq_question_type"],
        "retrieval_score": best["score"],
        "faq_hits_topk": faq_hits
    }


def retrieve_from_general_base(question: str, top_k: int = 3) -> dict:
    q_norm = normalize_for_match(question)
    q_emb = embed_text(q_norm)

    sem_sims = util.cos_sim(q_emb, general_embeddings)[0].cpu().numpy()
    tfidf_vec = general_vectorizer.transform([q_norm])
    tfidf_sims = cosine_similarity(tfidf_vec, general_tfidf)[0]

    scores = []
    for idx, row in general_df.iterrows():
        score = 0.58 * float(sem_sims[idx]) + 0.25 * float(tfidf_sims[idx]) + 0.17 * token_overlap_ratio(q_norm, row["search_text"])
        scores.append(score)

    top_idx = np.argsort(scores)[::-1][:top_k]

    general_hits = []
    for idx in top_idx:
        general_hits.append({
            "header": general_df.iloc[idx]["header"],
            "text": general_df.iloc[idx]["text"],
            "score": round(float(scores[idx]), 4)
        })

    if len(general_hits) == 0:
        return {
            "retrieval_status": "not_found",
            "retrieval_source": "Database-2.xlsx",
            "retrieval_type": "general_base",
            "matched_header": None,
            "matched_text": None,
            "retrieval_score": None,
            "general_hits_topk": []
        }

    best = general_hits[0]

    if float(best["score"]) < 0.28:
        return {
            "retrieval_status": "not_found",
            "retrieval_source": "Database-2.xlsx",
            "retrieval_type": "general_base",
            "matched_header": None,
            "matched_text": None,
            "retrieval_score": best["score"],
            "general_hits_topk": general_hits
        }

    return {
        "retrieval_status": "found",
        "retrieval_source": "Database-2.xlsx",
        "retrieval_type": "general_base",
        "matched_header": best["header"],
        "matched_text": best["text"],
        "retrieval_score": best["score"],
        "general_hits_topk": general_hits
    }

In [150]:
def retrieve_single_record(row: pd.Series) -> dict:
    route_target = row["route_target"]
    question = row["question"]

    if route_target == "blocked":
        return {
            "retrieval_status": "blocked",
            "retrieval_source": None,
            "retrieval_type": "blocked",
            "retrieval_score": None
        }

    if route_target == "out_of_scope":
        return {
            "retrieval_status": "out_of_scope",
            "retrieval_source": None,
            "retrieval_type": "out_of_scope",
            "retrieval_score": None
        }

    if route_target == "program_table":
        return retrieve_from_program_table(
            question=question,
            program_candidate=row.get("program_candidate"),
            program_candidates_top3=row.get("program_candidates_top3"),
            requested_field_best=row.get("requested_field_best"),
            field_candidates_topk=row.get("field_candidates_topk"),
            field_group=row.get("field_group"),
            unsupported_field_label=row.get("unsupported_field_label")
        )

    if route_target == "faq_base":
        return retrieve_from_faq(question=question, top_k=3)

    if route_target == "general_base":
        return retrieve_from_general_base(question=question, top_k=3)

    return {
        "retrieval_status": "unsupported_route",
        "retrieval_source": None,
        "retrieval_type": None,
        "retrieval_score": None
    }

In [151]:
def run_retrieval(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    retrieved_rows = []

    for _, row in df.iterrows():
        retrieval_result = retrieve_single_record(row)

        merged_row = row.to_dict()
        merged_row.update(retrieval_result)
        retrieved_rows.append(merged_row)

    return pd.DataFrame(retrieved_rows)

In [152]:
retrieved_df = run_retrieval(prepared_df_v3)

retrieved_df[
    [
        "question",
        "route_target",
        "retrieval_status",
        "retrieval_source",
        "matched_program",
        "matched_field",
        "matched_value",
        "matched_question",
        "matched_answer",
        "matched_header",
        "retrieval_score"
    ]
].head(20)

,question,route_target,retrieval_status,retrieval_source,matched_program,matched_field,matched_value,matched_question,matched_answer,matched_header,retrieval_score
0,Какие ЕГЭ нужны для поступления на программу «...,program_table,found,all_program.xlsx,анализ данных и искусственный интеллект,eges_budget,"обязательные егэ - русский язык: 70.0, математ...",NaN,NaN,NaN,0.5592
1,А на платное на «Анализ данных и искусственный...,program_table,found,all_program.xlsx,анализ данных и искусственный интеллект,eges_contract,"обязательные егэ - русский язык: 55, математик...",NaN,NaN,NaN,0.5628
2,Сколько стоит обучение на бизнес-информатике?,program_table,found,all_program.xlsx,бизнес-информатика,cost,435000,NaN,NaN,NaN,0.4230
3,Какой проходной балл был в 2024 году на бизнес...,program_table,found,all_program.xlsx,бизнес-информатика,pass_2024,281,NaN,NaN,NaN,0.3496
4,Сколько бюджетных мест в 2025 году на программ...,program_table,found,all_program.xlsx,внешняя политика и государственное управление,budget_2025,65,NaN,NaN,NaN,0.5608
5,Сколько платных мест на веб-разработке?,program_table,found,all_program.xlsx,веб-разработка,contract_2025,30,NaN,NaN,NaN,0.4025
6,Есть ли бюджет на программе «Веб-разработка»?,program_table,found,all_program.xlsx,веб-разработка,budget_2025,0,NaN,NaN,NaN,0.5575
7,Сколько лет учиться на программе «Внешнеэконом...,program_table,found,all_program.xlsx,внешнеэкономическая деятельность,edu_years,5.0,NaN,NaN,NaN,0.5637
8,К какому мегакластеру относится программа «Мар...,program_table,found,all_program.xlsx,маркетинг,megacluster,управление,NaN,NaN,NaN,0.5542
9,Какие требования по ЕГЭ на платное у программы...,program_table,found,all_program.xlsx,деловая журналистика,eges_contract,"обязательные егэ - русский язык: 40, литератур...",NaN,NaN,NaN,0.5588


# Формирование контекста

In [153]:
# =========================
# 7. БЛОК ФОРМИРОВАНИЯ КОНТЕКСТА ДЛЯ LLM
# =========================

import json
import pandas as pd


def stringify_value(value):
    """
    Аккуратно превращает значение в строку для контекста.
    """
    if value is None:
        return ""
    text = str(value).strip()
    return text


def make_not_found_context(question: str, route_target: str) -> dict:
    payload = {
        "question": question,
        "route_target": route_target,
        "found": False,
        "message": "Релевантная информация не найдена."
    }

    llm_context = f"""
Вопрос пользователя:
{question}

Маршрут:
{route_target}

Результат retrieval:
Релевантная информация не найдена.

Инструкция для ответа:
- Не придумывай данные.
- Честно скажи, что в найденных источниках нет достаточной информации для ответа.
- При необходимости предложи пользователю уточнить запрос.
""".strip()

    return {
        "context_status": "not_found",
        "context_type": route_target,
        "context_payload": payload,
        "llm_context": llm_context
    }

In [154]:
def build_out_of_scope_context(row: pd.Series) -> dict:
    question = row.get("question", "")

    payload = {
        "question": question,
        "route_target": "out_of_scope",
        "found": False,
        "message": "Вопрос не относится к тематике поступления."
    }

    llm_context = """
Запрос пользователя не относится к тематике поступления.
Нужно вежливо сообщить, что ассистент отвечает только на вопросы о поступлении:
программах, стоимости обучения, проходных баллах, ЕГЭ, количестве мест и мегакластерах.
""".strip()

    return {
        "context_status": "out_of_scope",
        "context_type": "out_of_scope",
        "context_payload": payload,
        "llm_context": llm_context
    }

In [155]:
def build_blocked_context(row: pd.Series) -> dict:
    question = row.get("question", "")

    payload = {
        "question": question,
        "route_target": "blocked",
        "found": False,
        "message": "Запрос заблокирован из-за некорректной или грубой формулировки."
    }

    llm_context = """
Запрос пользователя был заблокирован из-за некорректной или грубой формулировки.
Нужно ответить коротко и вежливо, попросив сформулировать вопрос корректно.
Не нужно обсуждать содержание грубого сообщения.
""".strip()

    return {
        "context_status": "blocked",
        "context_type": "blocked",
        "context_payload": payload,
        "llm_context": llm_context
    }

In [156]:
def build_program_table_context(row: pd.Series) -> dict:
    question = row["question"]
    matched_program = row.get("matched_program")
    field_values_found = row.get("field_values_found")
    retrieval_score = row.get("retrieval_score")

    if not isinstance(field_values_found, list):
        field_values_found = []

    payload = {
        "question": question,
        "route_target": "program_table",
        "found": True,
        "matched_program": matched_program,
        "field_values_found": field_values_found,
        "retrieval_score": retrieval_score
    }

    values_block = []
    for item in field_values_found:
        field_label = item.get("field_label", item.get("field_name", ""))
        value = stringify_value(item.get("value"))
        values_block.append(f"{field_label}: {value}")

    values_block_text = "\n".join(values_block) if values_block else "Подходящие значения не найдены."

    llm_context = f"""
Вопрос:
{question}

Программа:
{stringify_value(matched_program)}

Найденная информация:
{values_block_text}

Отвечай только по этой информации.
Если информации недостаточно, так и скажи.
""".strip()

    return {
        "context_status": "found",
        "context_type": "program_table",
        "context_payload": payload,
        "llm_context": llm_context
    }

In [157]:
def build_faq_context(row: pd.Series) -> dict:
    question = row["question"]
    matched_question = row.get("matched_question")
    matched_answer = row.get("matched_answer")
    retrieval_score = row.get("retrieval_score")
    faq_hits_topk = row.get("faq_hits_topk")

    if not isinstance(faq_hits_topk, list):
        faq_hits_topk = []

    payload = {
        "question": question,
        "route_target": "faq_base",
        "found": True,
        "matched_question": matched_question,
        "matched_answer": matched_answer,
        "retrieval_score": retrieval_score,
        "faq_hits_topk": faq_hits_topk
    }

    llm_context = f"""
Вопрос пользователя:
{question}

Похожий готовый вопрос:
{stringify_value(matched_question)}

Готовый ответ:
{stringify_value(matched_answer)}

Переформулируй ответ естественно и коротко, не добавляя новых фактов.
""".strip()

    return {
        "context_status": "found",
        "context_type": "faq_base",
        "context_payload": payload,
        "llm_context": llm_context
    }

In [158]:
def build_general_context(row: pd.Series) -> dict:
    question = row["question"]
    matched_header = row.get("matched_header")
    matched_text = row.get("matched_text")
    retrieval_score = row.get("retrieval_score")
    general_hits_topk = row.get("general_hits_topk")

    if not isinstance(general_hits_topk, list):
        general_hits_topk = []

    payload = {
        "question": question,
        "route_target": "general_base",
        "found": True,
        "matched_header": matched_header,
        "matched_text": matched_text,
        "retrieval_score": retrieval_score,
        "general_hits_topk": general_hits_topk
    }

    llm_context = f"""
Вопрос пользователя:
{question}

Подходящая информация:
Заголовок: {stringify_value(matched_header)}
Текст: {stringify_value(matched_text)}

Сформулируй короткий и понятный ответ только по этой информации.
""".strip()

    return {
        "context_status": "found",
        "context_type": "general_base",
        "context_payload": payload,
        "llm_context": llm_context
    }

In [159]:
def build_single_context(row: pd.Series) -> dict:
    route_target = row.get("route_target")
    retrieval_status = row.get("retrieval_status")

    if route_target == "blocked" or retrieval_status == "blocked":
        return build_blocked_context(row)

    if route_target == "out_of_scope" or retrieval_status == "out_of_scope":
        return build_out_of_scope_context(row)

    if retrieval_status == "found" or retrieval_status == "partial_found":
        if route_target == "program_table":
            return build_program_table_context(row)

        if route_target == "faq_base":
            return build_faq_context(row)

        if route_target == "general_base":
            return build_general_context(row)

    # Для not_found / program_not_found / field_not_found и т.д.
    payload = {
        "question": row.get("question", ""),
        "route_target": route_target,
        "found": False,
        "retrieval_status": retrieval_status,
        "matched_program": row.get("matched_program"),
        "unsupported_field_label": row.get("unsupported_field_label")
    }

    return {
        "context_status": retrieval_status if retrieval_status else "not_found",
        "context_type": route_target,
        "context_payload": payload,
        "llm_context": make_not_found_context(
            question=row.get("question", ""),
            route_target=route_target
        )["llm_context"]
    }

In [160]:
def run_context_building(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    output_rows = []

    for _, row in df.iterrows():
        context_result = build_single_context(row)

        merged_row = row.to_dict()
        merged_row.update(context_result)
        output_rows.append(merged_row)

    return pd.DataFrame(output_rows)

In [161]:
context_df = run_context_building(retrieved_df)

context_df[
    [
        "question",
        "route_target",
        "retrieval_status",
        "context_status",
        "context_type",
        "llm_context"
    ]
].head(10)

,question,route_target,retrieval_status,context_status,context_type,llm_context
0,Какие ЕГЭ нужны для поступления на программу «...,program_table,found,found,program_table,Вопрос:\nКакие ЕГЭ нужны для поступления на пр...
1,А на платное на «Анализ данных и искусственный...,program_table,found,found,program_table,Вопрос:\nА на платное на «Анализ данных и иску...
2,Сколько стоит обучение на бизнес-информатике?,program_table,found,found,program_table,Вопрос:\nСколько стоит обучение на бизнес-инфо...
3,Какой проходной балл был в 2024 году на бизнес...,program_table,found,found,program_table,Вопрос:\nКакой проходной балл был в 2024 году ...
4,Сколько бюджетных мест в 2025 году на программ...,program_table,found,found,program_table,Вопрос:\nСколько бюджетных мест в 2025 году на...
5,Сколько платных мест на веб-разработке?,program_table,found,found,program_table,Вопрос:\nСколько платных мест на веб-разработк...
6,Есть ли бюджет на программе «Веб-разработка»?,program_table,found,found,program_table,Вопрос:\nЕсть ли бюджет на программе «Веб-разр...
7,Сколько лет учиться на программе «Внешнеэконом...,program_table,found,found,program_table,Вопрос:\nСколько лет учиться на программе «Вне...
8,К какому мегакластеру относится программа «Мар...,program_table,found,found,program_table,Вопрос:\nК какому мегакластеру относится прогр...
9,Какие требования по ЕГЭ на платное у программы...,program_table,found,found,program_table,Вопрос:\nКакие требования по ЕГЭ на платное у ...


# Блок LLM

In [162]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Русскоязычная instruct-модель
ANSWER_MODEL_NAME = "Vikhrmodels/Vikhr-Qwen-2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer_answer = AutoTokenizer.from_pretrained(
    ANSWER_MODEL_NAME,
    trust_remote_code=True
)

# Вариант без 4-bit квантования — проще и надежнее
model_answer = AutoModelForCausalLM.from_pretrained(
    ANSWER_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True
)

model_answer.eval()

Device: cpu


Loading weights: 100%|██████████| 338/338 [00:07<00:00, 44.86it/s] 
Some parameters are on the meta device because they were offloaded to the disk.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151665, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [163]:
SYSTEM_PROMPT_ANSWER = """
Ты — русскоязычный помощник для абитуриентов университета.

Твоя задача:
- отвечать только по переданной информации;
- не выдумывать факты, числа, требования, баллы, стоимость, места и названия программ;
- если информации недостаточно, честно говорить об этом;
- отвечать естественно и коротко.

Строгие запреты:
- не упоминай слова и фразы: "контекст", "маршрут", "retrieval", "program_table", "faq_base", "general_base";
- не упоминай технические названия полей и колонок;
- не начинай ответ с фраз:
  "согласно...",
  "на основе...",
  "по имеющейся информации...",
  "исходя из...",
  "можно сделать вывод...",
  "можно выделить...";
- не добавляй вводные служебные фразы;
- не пиши общие рассуждения, советы, пояснения и предупреждения, если их нет в данных;
- не дописывай фразы вроде "обратите внимание", "в зависимости от уровня знаний", "вам потребуется", если этого нет в найденной информации.

Требования к стилю:
- отвечай на русском языке;
- начинай сразу с сути;
- пиши так, как будто отвечаешь человеку в чате;
- для фактологических вопросов давай короткий прямой ответ;
- не используй markdown-таблицы.
""".strip()

In [164]:
def fallback_not_found_answer(row: pd.Series) -> str:
    return "Я не нашёл достаточной информации в базе, чтобы надёжно ответить на этот вопрос."


def fallback_program_table_answer(row: pd.Series) -> str:
    matched_program = row.get("matched_program")
    field_values_found = row.get("field_values_found")

    if not isinstance(field_values_found, list) or len(field_values_found) == 0:
        if matched_program:
            return f"Я нашёл программу «{matched_program}», но нужное значение в таблице определить не удалось."
        return "Я не смог надёжно извлечь нужное значение из таблицы."

    if len(field_values_found) == 1:
        item = field_values_found[0]
        label = item.get("field_label", item.get("field_name", ""))
        value = str(item.get("value", "")).strip()

        if matched_program:
            return f"Для программы «{matched_program}» {label.lower()} — {value}."
        return f"{label} — {value}."

    parts = []
    if matched_program:
        parts.append(f"Для программы «{matched_program}» найдено несколько связанных значений:")

    for item in field_values_found:
        label = item.get("field_label", item.get("field_name", ""))
        value = str(item.get("value", "")).strip()
        parts.append(f"{label}: {value}")

    return " ".join(parts)


def fallback_faq_answer(row: pd.Series) -> str:
    matched_answer = row.get("matched_answer")
    if matched_answer and str(matched_answer).strip():
        return str(matched_answer).strip()
    return "Я не смог найти подходящий готовый ответ в FAQ."


def fallback_general_answer(row: pd.Series) -> str:
    matched_text = row.get("matched_text")
    if matched_text and str(matched_text).strip():
        return str(matched_text).strip()
    return "Я не смог найти подходящий фрагмент в базе знаний."

In [165]:
def build_answer_prompt(row: pd.Series) -> str:
    question = row.get("question", "")
    route_target = row.get("route_target", "")
    llm_context = row.get("llm_context", "")
    field_group = row.get("field_group", "")

    if route_target == "program_table":
        return f"""
Вопрос пользователя:
{question}

Информация для ответа:
{llm_context}

Нужно дать короткий и естественный ответ пользователю.

Правила:
- отвечай только по информации выше;
- начинай сразу с ответа, без вводных фраз;
- не пиши "согласно...", "на основе...", "по информации...", "исходя из...";
- не добавляй общие фразы вроде "вам потребуется набрать определённое количество баллов";
- не добавляй советы, комментарии и пояснения, которых нет в информации;
- если найдено несколько значений, просто перечисли их естественно;
- если информации недостаточно, скажи об этом прямо.

Примеры желаемого стиля:
Вопрос: Сколько стоит обучение на бизнес-информатике?
Ответ: Обучение на программе «Бизнес-информатика» стоит 435000 рублей.

Вопрос: Какие ЕГЭ нужны для поступления на анализ данных и искусственный интеллект?
Ответ: Для программы «Анализ данных и искусственный интеллект» требования указаны отдельно. Для бюджета: русский язык — 60, математика — 50. Для платного обучения: русский язык — 70, математика — 62.

Теперь сформулируй ответ пользователю.
""".strip()

    if route_target == "faq_base":
        return f"""
Вопрос пользователя:
{question}

Информация для ответа:
{llm_context}

Сформулируй короткий и естественный ответ пользователю.

Правила:
- начинай сразу с ответа;
- не пиши "согласно..." и другие служебные вводные фразы;
- можно переформулировать текст, но нельзя добавлять новые факты.
""".strip()

    if route_target == "general_base":
        return f"""
Вопрос пользователя:
{question}

Информация для ответа:
{llm_context}

Сформулируй короткий и понятный ответ пользователю.

Правила:
- начинай сразу с ответа;
- не пиши "согласно..." и другие служебные вводные фразы;
- не добавляй факты, которых нет в информации.
""".strip()

    return f"""
Вопрос пользователя:
{question}

Информация для ответа:
{llm_context}

Сформулируй короткий и естественный ответ пользователю без служебных вводных фраз.
""".strip()

In [166]:
def answer_llm(system_prompt: str,
               user_prompt: str,
               max_new_tokens: int = 160,
               temperature: float = 0.0) -> str | None:
    try:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

        prompt_text = tokenizer_answer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer_answer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        )

        inputs = {k: v.to(model_answer.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model_answer.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                pad_token_id=tokenizer_answer.eos_token_id
            )

        generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer_answer.decode(generated_ids, skip_special_tokens=True).strip()

        if not answer:
            return None

        return answer

    except Exception as e:
        print("LLM generation error:", e)
        return None

In [167]:
def build_fallback_answer(row: pd.Series) -> str:
    context_status = row.get("context_status")
    route_target = row.get("route_target")

    if route_target == "blocked" or context_status == "blocked":
        return get_blocked_response()

    if route_target == "out_of_scope" or context_status == "out_of_scope":
        return get_out_of_scope_response()

    if context_status != "found":
        return fallback_not_found_answer(row)

    if route_target == "program_table":
        return fallback_program_table_answer(row)

    if route_target == "faq_base":
        return fallback_faq_answer(row)

    if route_target == "general_base":
        return fallback_general_answer(row)

    return fallback_not_found_answer(row)

In [168]:
import re

def clean_final_answer(text: str) -> str:
    if not text:
        return text

    cleaned = text.strip()

    bad_starts = [
        r"^согласно предоставленной информации[,:\s-]*",
        r"^согласно имеющейся информации[,:\s-]*",
        r"^согласно представленным данным[,:\s-]*",
        r"^по имеющейся информации[,:\s-]*",
        r"^на основе предоставленной информации[,:\s-]*",
        r"^на основе представленной информации[,:\s-]*",
        r"^исходя из представленных данных[,:\s-]*",
        r"^в предоставленных материалах[,:\s-]*",
        r"^можно сделать вывод[,:\s-]*что\s*",
        r"^можно выделить следующие[,:\s-]*",
    ]

    for pattern in bad_starts:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    bad_phrases = [
        r"вам потребуется набрать определ[её]нное количество баллов на егэ\.?",
        r"обратите внимание[,:\s-]*",
        r"в зависимости от вашего текущего уровня знаний.*$",
        r"возможно[, ]+вам потреб.*$",
        r"это минимальные необходимые.*$",
    ]

    for pattern in bad_phrases:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # если модель начала с двоеточия или тире после очистки
    cleaned = re.sub(r"^[:\-–—\s]+", "", cleaned).strip()

    if cleaned:
        cleaned = cleaned[0].upper() + cleaned[1:]

    return cleaned

In [169]:
def generate_single_answer(row: pd.Series) -> dict:
    """
    Ключевое изменение:
    - для структурированных и retrieval-ответов не даём LLM свободу переписывать факты;
    - используем извлечённый / шаблонный ответ напрямую.
    Это сильно уменьшает галлюцинации на стоимости, ЕГЭ, местах и на вопросах,
    где в данных ответа вообще нет.
    """
    if row.get("route_target") == "blocked" or row.get("context_status") == "blocked":
        final_answer = get_blocked_response()
        return {
            "answer_prompt": None,
            "final_answer": final_answer,
            "generation_mode": "blocked_template"
        }

    if row.get("route_target") == "out_of_scope" or row.get("context_status") == "out_of_scope":
        final_answer = get_out_of_scope_response()
        return {
            "answer_prompt": None,
            "final_answer": final_answer,
            "generation_mode": "out_of_scope_template"
        }

    # Для всех retrieval-веток используем безопасный детерминированный ответ.
    final_answer = build_fallback_answer(row)
    final_answer = clean_final_answer(final_answer)

    return {
        "answer_prompt": None,
        "final_answer": final_answer,
        "generation_mode": "deterministic_fallback"
    }

In [170]:
def run_answer_generation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    output_rows = []

    for _, row in df.iterrows():
        answer_result = generate_single_answer(row)

        merged_row = row.to_dict()
        merged_row.update(answer_result)
        output_rows.append(merged_row)

    return pd.DataFrame(output_rows)

In [171]:
answer_df = run_answer_generation(context_df)

answer_df[
    [
        "question",
        "route_target",
        "context_status",
        "generation_mode",
        "final_answer"
    ]
].head(15)

,question,route_target,context_status,generation_mode,final_answer
0,Какие ЕГЭ нужны для поступления на программу «...,program_table,found,deterministic_fallback,Для программы «анализ данных и искусственный и...
1,А на платное на «Анализ данных и искусственный...,program_table,found,deterministic_fallback,Для программы «анализ данных и искусственный и...
2,Сколько стоит обучение на бизнес-информатике?,program_table,found,deterministic_fallback,Для программы «бизнес-информатика» стоимость о...
3,Какой проходной балл был в 2024 году на бизнес...,program_table,found,deterministic_fallback,Для программы «бизнес-информатика» проходной б...
4,Сколько бюджетных мест в 2025 году на программ...,program_table,found,deterministic_fallback,Для программы «внешняя политика и государствен...
5,Сколько платных мест на веб-разработке?,program_table,found,deterministic_fallback,Для программы «веб-разработка» платные места —...
6,Есть ли бюджет на программе «Веб-разработка»?,program_table,found,deterministic_fallback,Для программы «веб-разработка» бюджетные места...
7,Сколько лет учиться на программе «Внешнеэконом...,program_table,found,deterministic_fallback,Для программы «внешнеэкономическая деятельност...
8,К какому мегакластеру относится программа «Мар...,program_table,found,deterministic_fallback,Для программы «маркетинг» мегакластер — управл...
9,Какие требования по ЕГЭ на платное у программы...,program_table,found,deterministic_fallback,Для программы «деловая журналистика» егэ для п...


# Блок проверки ответа

In [172]:
STRICT_FIELD_GROUPS = {"cost", "pass", "places", "eges"}

In [173]:
def normalize_light(text: str) -> str:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_digits(text: str) -> list:
    """
    Извлекаем числовые фрагменты из текста.
    Это мягкая эвристика для проверки, что в ответе есть числа,
    если вопрос был про стоимость, баллы, места и т.д.
    """
    if text is None:
        return []
    return re.findall(r"\d+[.,]?\d*", str(text))


def value_fragment_present(answer: str, value: str) -> bool:
    """
    Проверяем, присутствует ли в ответе хотя бы заметный фрагмент найденного значения.
    """
    if not answer or not value:
        return False

    answer_norm = normalize_light(answer)
    value_norm = normalize_light(value)

    # Полное вхождение
    if value_norm in answer_norm:
        return True

    # Для длинных текстов ищем первые осмысленные куски
    if len(value_norm) >= 12 and value_norm[:20] in answer_norm:
        return True

    # Для чисел проверяем пересечение по числовым токенам
    value_digits = set(extract_digits(value_norm))
    answer_digits = set(extract_digits(answer_norm))
    if value_digits and (value_digits & answer_digits):
        return True

    return False

In [174]:
def normalize_light(text: str) -> str:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_digits(text: str) -> list:
    """
    Извлекаем числовые фрагменты из текста.
    Это мягкая эвристика для проверки, что в ответе есть числа,
    если вопрос был про стоимость, баллы, места и т.д.
    """
    if text is None:
        return []
    return re.findall(r"\d+[.,]?\d*", str(text))


def value_fragment_present(answer: str, value: str) -> bool:
    """
    Проверяем, присутствует ли в ответе хотя бы заметный фрагмент найденного значения.
    """
    if not answer or not value:
        return False

    answer_norm = normalize_light(answer)
    value_norm = normalize_light(value)

    # Полное вхождение
    if value_norm in answer_norm:
        return True

    # Для длинных текстов ищем первые осмысленные куски
    if len(value_norm) >= 12 and value_norm[:20] in answer_norm:
        return True

    # Для чисел проверяем пересечение по числовым токенам
    value_digits = set(extract_digits(value_norm))
    answer_digits = set(extract_digits(answer_norm))
    if value_digits and (value_digits & answer_digits):
        return True

    return False

In [175]:
def check_program_table_grounding(row: pd.Series, answer: str) -> list:
    """
    Проверка, что ответ по таблице реально опирается на найденные данные.
    """
    issues = []

    field_group = row.get("field_group")
    field_values_found = row.get("field_values_found")
    retrieval_status = row.get("retrieval_status")
    matched_program = row.get("matched_program")
    requested_field_best = row.get("requested_field_best")
    field_candidates_topk = row.get("field_candidates_topk")

    if retrieval_status not in {"found", "partial_found"}:
        issues.append("no_retrieval_result")
        return issues

    if not isinstance(field_values_found, list):
        field_values_found = []

    answer_norm = normalize_light(answer)

    # 1. Для строгих полей нужно, чтобы ответ хоть как-то пересекался с найденными значениями
    if field_group in STRICT_FIELD_GROUPS:
        grounded = False
        for item in field_values_found:
            value = str(item.get("value", "")).strip()
            if value_fragment_present(answer, value):
                grounded = True
                break

        if not grounded and len(field_values_found) > 0:
            issues.append("answer_not_grounded_in_found_values")

    # 2. Если программа найдена, желательно чтобы она была названа в ответе
    if matched_program:
        prog_norm = normalize_light(matched_program)
        if prog_norm and prog_norm not in answer_norm:
            issues.append("program_name_not_mentioned")

    # 3. Если есть несколько найденных значений, а ответ выглядит как однозначный,
    # это может быть проблемой
    if len(field_values_found) >= 2:
        if field_group in {"eges", "places"}:
            mentioned_count = 0
            for item in field_values_found:
                value = str(item.get("value", "")).strip()
                if value_fragment_present(answer, value):
                    mentioned_count += 1

            if mentioned_count < 2:
                issues.append("ambiguous_values_not_reflected")

    return issues

In [176]:
def check_faq_grounding(row: pd.Series, answer: str) -> list:
    issues = []

    matched_answer = str(row.get("matched_answer", "")).strip()
    retrieval_status = row.get("retrieval_status")

    if retrieval_status != "found":
        issues.append("no_retrieval_result")
        return issues

    # Для FAQ не требуем почти буквального совпадения,
    # но если ответ совсем пустой или слишком далёкий по длине, это подозрительно.
    if not str(answer).strip():
        issues.append("empty_answer")

    if matched_answer and len(str(answer).strip()) < 5:
        issues.append("answer_too_short_for_faq")

    return issues


def check_general_grounding(row: pd.Series, answer: str) -> list:
    issues = []

    matched_text = str(row.get("matched_text", "")).strip()
    retrieval_status = row.get("retrieval_status")

    if retrieval_status != "found":
        issues.append("no_retrieval_result")
        return issues

    if not str(answer).strip():
        issues.append("empty_answer")

    return issues

In [177]:
def build_safe_answer_after_validation(row: pd.Series) -> str:
    """
    Если ответ признан ненадёжным, возвращаем безопасный fallback.
    """
    route_target = row.get("route_target")

    if route_target == "program_table":
        return fallback_program_table_answer(row)

    if route_target == "faq_base":
        return fallback_faq_answer(row)

    if route_target == "general_base":
        return fallback_general_answer(row)

    return fallback_not_found_answer(row)

In [178]:
def validate_single_answer(row: pd.Series) -> dict:
    if row.get("route_target") == "blocked":
        return {
            "answer_validation_status": "passed_blocked_template",
            "answer_issues": [],
            "final_answer_checked": row.get("final_answer", get_blocked_response()),
            "answer_safe_to_show": True
        }

    if row.get("route_target") == "out_of_scope":
        return {
            "answer_validation_status": "passed_out_of_scope_template",
            "answer_issues": [],
            "final_answer_checked": row.get("final_answer", get_out_of_scope_response()),
            "answer_safe_to_show": True
        }

    raw_answer = str(row.get("final_answer", "")).strip()
    route_target = row.get("route_target")

    issues = []

    if route_target == "program_table":
        issues = check_program_table_grounding(row, raw_answer)

    elif route_target == "faq_base":
        issues = check_faq_grounding(row, raw_answer)

    elif route_target == "general_base":
        issues = check_general_grounding(row, raw_answer)

    severe_issue_types = {
        "no_retrieval_result",
        "answer_not_grounded_in_found_values",
        "ambiguous_values_not_reflected",
        "empty_answer"
    }

    severe_found = any(issue in severe_issue_types for issue in issues)

    if severe_found:
        final_answer_checked = build_safe_answer_after_validation(row)
        answer_validation_status = "replaced_with_safe_fallback"
        answer_safe_to_show = True
    else:
        final_answer_checked = raw_answer
        answer_validation_status = "passed"
        answer_safe_to_show = True

    return {
        "answer_validation_status": answer_validation_status,
        "answer_issues": issues,
        "final_answer_checked": final_answer_checked,
        "answer_safe_to_show": answer_safe_to_show
    }

In [179]:
def run_answer_validation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    output_rows = []

    for _, row in df.iterrows():
        validation_result = validate_single_answer(row)

        merged_row = row.to_dict()
        merged_row.update(validation_result)
        output_rows.append(merged_row)

    return pd.DataFrame(output_rows)

In [180]:
validated_df = run_answer_validation(answer_df)

validated_df[
    [
        "question",
        "route_target",
        "generation_mode",
        "final_answer",
        "answer_validation_status",
        "answer_issues",
        "final_answer_checked"
    ]
].head(20)

,question,route_target,generation_mode,final_answer,answer_validation_status,answer_issues,final_answer_checked
0,Какие ЕГЭ нужны для поступления на программу «...,program_table,deterministic_fallback,Для программы «анализ данных и искусственный и...,passed,[],Для программы «анализ данных и искусственный и...
1,А на платное на «Анализ данных и искусственный...,program_table,deterministic_fallback,Для программы «анализ данных и искусственный и...,passed,[],Для программы «анализ данных и искусственный и...
2,Сколько стоит обучение на бизнес-информатике?,program_table,deterministic_fallback,Для программы «бизнес-информатика» стоимость о...,passed,[],Для программы «бизнес-информатика» стоимость о...
3,Какой проходной балл был в 2024 году на бизнес...,program_table,deterministic_fallback,Для программы «бизнес-информатика» проходной б...,passed,[],Для программы «бизнес-информатика» проходной б...
4,Сколько бюджетных мест в 2025 году на программ...,program_table,deterministic_fallback,Для программы «внешняя политика и государствен...,passed,[],Для программы «внешняя политика и государствен...
5,Сколько платных мест на веб-разработке?,program_table,deterministic_fallback,Для программы «веб-разработка» платные места —...,passed,[],Для программы «веб-разработка» платные места —...
6,Есть ли бюджет на программе «Веб-разработка»?,program_table,deterministic_fallback,Для программы «веб-разработка» бюджетные места...,passed,[],Для программы «веб-разработка» бюджетные места...
7,Сколько лет учиться на программе «Внешнеэконом...,program_table,deterministic_fallback,Для программы «внешнеэкономическая деятельност...,passed,[],Для программы «внешнеэкономическая деятельност...
8,К какому мегакластеру относится программа «Мар...,program_table,deterministic_fallback,Для программы «маркетинг» мегакластер — управл...,passed,[],Для программы «маркетинг» мегакластер — управл...
9,Какие требования по ЕГЭ на платное у программы...,program_table,deterministic_fallback,Для программы «деловая журналистика» егэ для п...,passed,[],Для программы «деловая журналистика» егэ для п...


# fallback
Если система ничего не нашла, она не должна фантазировать. 
Она должна честно сказать что-то вроде:    
•	«Я не нашёл точной информации по этому вопросу в базе данных.»     
•	«Уточните, пожалуйста, название программы.»      
•	«В базе нет полного ответа на этот вопрос.»     
Это очень важная часть хорошего ассистента:      
лучше честно сказать “не найдено”, чем придумать неправильный ответ.     

In [181]:
def get_clarification_program_response(row: pd.Series) -> str:
    candidate_programs = row.get("program_candidates_top3") or row.get("program_candidates")
    if isinstance(candidate_programs, list) and len(candidate_programs) > 0:
        top_candidates = []
        for item in candidate_programs[:3]:
            if isinstance(item, dict):
                top_candidates.append(str(item.get("program", "")))
            else:
                top_candidates.append(str(item))
        top_candidates = [x for x in top_candidates if x]
        if top_candidates:
            joined = "; ".join(top_candidates)
            return (
                "Я не смог однозначно определить программу. "
                f"Уточните, пожалуйста, название программы. Возможно, вы имели в виду: {joined}."
            )

    return (
        "Уточните, пожалуйста, название программы, чтобы я смог найти точную информацию "
        "по стоимости, ЕГЭ, проходным баллам или количеству мест."
    )

In [182]:
def get_clarification_field_response(row: pd.Series) -> str:
    matched_program = row.get("matched_program")
    if matched_program:
        return (
            f"Я нашёл программу «{matched_program}», но не смог точно определить, "
            "какой именно показатель вы хотите узнать. Уточните, пожалуйста, вопрос."
        )

    return (
        "Я не смог точно определить, какое именно значение нужно найти. "
        "Уточните, пожалуйста, вопрос."
    )

In [183]:
def fallback_not_found_answer(row: pd.Series) -> str:
    question_type = row.get("question_type")
    route_target = row.get("route_target")

    if route_target == "program_table" or question_type == "table":
        return (
            "Я не нашёл точной информации по этому запросу в таблице программ. "
            "Уточните, пожалуйста, название программы или параметр вопроса."
        )

    if route_target == "faq_base" or question_type == "faq":
        return "Я не нашёл подходящего готового ответа на этот вопрос в FAQ."

    if route_target == "general_base" or question_type == "general":
        return "Я не нашёл полного ответа на этот вопрос в текстовой базе знаний."

    return "Я не нашёл достаточной информации в базе, чтобы надёжно ответить на этот вопрос."

In [184]:
def fallback_program_table_answer(row: pd.Series) -> str:
    matched_program = row.get("matched_program")
    field_values_found = row.get("field_values_found")
    context_status = row.get("context_status")
    unsupported_field_label = row.get("unsupported_field_label")
    matched_field = row.get("matched_field")
    matched_value = row.get("matched_value")

    if context_status == "ambiguous_program" or context_status == "program_not_found":
        return get_clarification_program_response(row)

    if context_status == "ambiguous_field":
        return get_clarification_field_response(row)

    if unsupported_field_label:
        if matched_program:
            return (
                f"Я нашёл программу «{matched_program}», но в доступных данных нет показателя "
                "по вашему запросу."
            )
        return "В доступных данных нет показателя по вашему запросу."

    if not isinstance(field_values_found, list):
        field_values_found = []

    # Специальный случай: бюджет = 0 / не было бюджета
    if matched_field == "budget_2025":
        try:
            value_num = float(str(matched_value).replace(",", "."))
        except Exception:
            value_num = None

        pass_text = str(row.get("matched_value", "")).lower()
        if value_num == 0 or "не было бюджета" in str(row.get("pass_2024", "")).lower():
            if matched_program:
                return f"На программе «{matched_program}» бюджетных мест нет."
            return "Бюджетных мест нет."

    if context_status == "field_not_found":
        if matched_program:
            return (
                f"Я нашёл программу «{matched_program}», но в доступных данных нет точного значения "
                "по вашему запросу."
            )
        return "Я не смог найти в таблице точное значение по вашему запросу."

    if len(field_values_found) == 0:
        if matched_program:
            return (
                f"Я нашёл программу «{matched_program}», но нужное значение "
                "в таблице определить не удалось."
            )
        return "Я не смог надёжно извлечь нужное значение из таблицы."

    if len(field_values_found) == 1:
        item = field_values_found[0]
        label = item.get("field_label", item.get("field_name", ""))
        value = str(item.get("value", "")).strip()

        # Отдельно обрабатываем вопрос "есть ли бюджет"
        if item.get("field_name") == "budget_2025":
            try:
                value_num = float(str(value).replace(",", "."))
            except Exception:
                value_num = None

            if value_num == 0:
                if matched_program:
                    return f"На программе «{matched_program}» бюджетных мест нет."
                return "Бюджетных мест нет."

        if matched_program:
            return f"Для программы «{matched_program}» {label.lower()} — {value}."
        return f"{label} — {value}."

    parts = []
    if matched_program:
        parts.append(f"Для программы «{matched_program}» найдено несколько связанных значений:")
    else:
        parts.append("Найдено несколько связанных значений:")

    for item in field_values_found:
        label = item.get("field_label", item.get("field_name", ""))
        value = str(item.get("value", "")).strip()
        parts.append(f"{label}: {value}")

    parts.append("Если нужно, уточните, какое именно значение вас интересует.")
    return " ".join(parts)

In [185]:
def fallback_faq_answer(row: pd.Series) -> str:
    matched_answer = row.get("matched_answer")

    if matched_answer and str(matched_answer).strip():
        return str(matched_answer).strip()

    return "Я не смог найти подходящий готовый ответ в FAQ."

In [186]:
def fallback_general_answer(row: pd.Series) -> str:
    matched_text = row.get("matched_text")

    if matched_text and str(matched_text).strip():
        return str(matched_text).strip()

    return "Я не смог найти подходящий фрагмент в базе знаний."

In [187]:
def build_fallback_answer(row: pd.Series) -> str:
    context_status = row.get("context_status")
    route_target = row.get("route_target")

    if route_target == "blocked" or context_status == "blocked":
        return get_blocked_response()

    if route_target == "out_of_scope" or context_status == "out_of_scope":
        return get_out_of_scope_response()

    if route_target == "program_table":
        return fallback_program_table_answer(row)

    if route_target == "faq_base":
        return fallback_faq_answer(row)

    if route_target == "general_base":
        return fallback_general_answer(row)

    return fallback_not_found_answer(row)

In [188]:
validated_df["fallback_answer"] = validated_df.apply(build_fallback_answer, axis=1)

In [189]:
validated_df[["question", 'route_target', "final_answer_checked", "fallback_answer"]].to_excel('validated_df.xlsx')